# RSNA Phase B: read-only real-data and DICOM audit

Attach only the official competition input. Select CPU / no accelerator and Internet Off, then Run All from a clean session. Source hashes are checked before the single validated configuration, CSV preflight, bounded Phase B audit, and JSON write. No installation, credentials, repository imports, dataset copies, predictions, or training occur.

Only an already installed pydicom can read headers or run up to 12 guarded pixel smoke tests. Missing dependencies and exhausted budgets remain BLOCKED or NOT_ASSESSED. Series, header, and decoding samples never certify full DICOM coverage or all codecs. Slice ordering/gaps use position projected onto orientation; SliceThickness is not a spacing substitute. No raw reports, patient identifiers, UID tables, pixels, or arrays are retained. Sparse labels require a separate validation review; training_ready always remains false. Visible test studies describe this runtime sample, not the hidden evaluation set.


In [ ]:
# Reviewed source bundle: no filesystem extraction or network access.
import hashlib
import importlib.abc
import importlib.util
import json
import sys

_RSNA_BUNDLE = json.loads('{"sha256":{"__init__.py":"f7c27acdb5e30e9734b6fbb6d55809ec8bcfef85c15cbac858331d23941b994e","audit.py":"13bcb1acc910af81000805e1e32d139475fc7b97cbdcf5cd4d8d6320731cfbd5","cli.py":"64f2631094bcc8aab879b2ce4471731aa3fd7d39add1bbea5f019221c73ceff6","config.py":"6dbb61af824770c3ac0dca64636762f14f9cc71e3c3bd0193b200211996347cd","contracts.py":"34d3892dd8fe63dde0f5494815240b008082406b94e6fff0d16528b45c3fda05","dicom_audit.py":"d0f113663292b5e083631c0c5db84440fbfcf062151c22f5f6494370c1a1d215","inputs.py":"48fa4fe5c0c5f183e6975113cfd889ae2c8c9139e9b98b944dc3708e362f5d8e","metrics.py":"4121fb00bd7e2666b3f0885d32d1430d7e102d9ee8d70d4f7bf8d27dbea2f774","phase_b.py":"ec506c0ae3cbd8b22066f1717046e9e7c3593b2a4124a0b825020ec93523dee7","phase_b_config.py":"e1e33517bcef7bb49b7967805ff13526fdd33d3188b0cfbf76787cd57446aadc","report_quality.py":"6f97c131f021db422e7009b9f8e2f8aa26396782469c0e5ba97e77a47337bf26","series_inventory.py":"13e5edac10de26231f99c8672da26b41382544af28c64c3b1b0e0177883150dc","submission.py":"9dc8941373de54bfa2ebe037e485d0060baea653bec6c981aa6a447c7673d5b0","validation_support.py":"944e923386f824ef2c3b8f56150c7c2ee78306c8813abf43367c52cc9d09b312"},"sources":{"__init__":"\\"\\"\\"Read-only audit software; DICOM support and model performance require evidence.\\"\\"\\"\\r\\n\\r\\n__version__ = \\"0.1.0\\"\\r\\n","audit":"\\"\\"\\"Initial preflight contracts only; never a Phase B or training approval.\\"\\"\\"\\r\\n\\r\\nfrom datetime import datetime, timezone\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\nimport platform\\r\\nimport time\\r\\nimport uuid\\r\\n\\r\\nfrom .config import AuditConfig\\r\\nfrom .contracts import ContractError, TARGETS, audit_series, audit_studies\\r\\nfrom .inputs import InputError, MANIFEST_NAMES, discover_inputs, read_csv_manifest, safe_absolute\\r\\nfrom .submission import validate_submission\\r\\n\\r\\n\\r\\ndef _summarize(contract: dict) -> dict:\\r\\n    # Keep outputs compact; no full identifier tables, report text, or pixels.\\r\\n    return {key: value for key, value in contract.items() if key not in (\\"study_ids\\", \\"series_ids\\")}\\r\\n\\r\\n\\r\\ndef _csv_contracts(root: str, config: AuditConfig, progress: dict) -> dict:\\r\\n    tables = {}\\r\\n    columns = {}\\r\\n    hashes = {}\\r\\n    for name in MANIFEST_NAMES:\\r\\n        progress[\\"attempted_files\\"].append(name)\\r\\n        columns[name], tables[name], hashes[name] = read_csv_manifest(Path(root) / name, config)\\r\\n        progress[\\"parsed_files\\"].append(name)\\r\\n        progress[\\"sha256\\"][name] = hashes[name]\\r\\n    train = audit_studies(tables[\\"train.csv\\"], \\"train\\")\\r\\n    test = audit_studies(tables[\\"test.csv\\"], \\"test\\")\\r\\n    train_ids, test_ids = set(train[\\"study_ids\\"]), set(test[\\"study_ids\\"])\\r\\n    overlap = train_ids & test_ids\\r\\n    if overlap:\\r\\n        raise ContractError(\\"Train/test study identifiers overlap\\")\\r\\n    train_series = audit_series(tables[\\"train_series.csv\\"], train_ids)\\r\\n    test_series = audit_series(tables[\\"test_series.csv\\"], test_ids)\\r\\n    series_overlap = set(train_series[\\"series_ids\\"]) & set(test_series[\\"series_ids\\"])\\r\\n    if series_overlap:\\r\\n        raise ContractError(\\"Train/test series identifiers overlap\\")\\r\\n    missing_train = train_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"train_series.csv\\"]}\\r\\n    missing_test = test_ids - {row[\\"StudyInstanceUID\\"] for row in tables[\\"test_series.csv\\"]}\\r\\n    if missing_train or missing_test:\\r\\n        raise ContractError(\\"Study manifest includes studies without a series reference\\")\\r\\n    sample = []\\r\\n    for row in tables[\\"sample_submission.csv\\"]:\\r\\n        converted = dict(row)\\r\\n        for target in TARGETS:\\r\\n            if target not in converted:\\r\\n                raise ContractError(\\"Sample submission is missing target columns\\")\\r\\n            try:\\r\\n                converted[target] = float(converted[target])\\r\\n            except (TypeError, ValueError) as error:\\r\\n                raise ContractError(\\"Sample submission scores must be numeric\\") from error\\r\\n        sample.append(converted)\\r\\n    submission = validate_submission(sample, test_ids, columns=columns[\\"sample_submission.csv\\"])\\r\\n    valid = all(result[\\"status\\"] == \\"PASS\\" for result in (train, test, train_series, test_series))\\r\\n    return {\\r\\n        \\"status\\": \\"VALID\\" if valid else \\"INVALID\\", \\"columns\\": columns,\\r\\n        \\"sha256\\": hashes, \\"train\\": _summarize(train), \\"test\\": _summarize(test),\\r\\n        \\"train_series\\": _summarize(train_series), \\"test_series\\": _summarize(test_series),\\r\\n        \\"sample_submission\\": submission, \\"train_test_study_overlap_count\\": 0,\\r\\n        \\"train_test_series_overlap_count\\": 0,\\r\\n        \\"scope\\": \\"CSV contracts only; no patient, report-quality, geometry or image-integrity audit\\",\\r\\n    }\\r\\n\\r\\n\\r\\ndef run_preflight(config: AuditConfig, source_identity: dict | None = None) -> dict:\\r\\n    if not isinstance(config, AuditConfig):\\r\\n        raise TypeError(\\"run_preflight requires validated AuditConfig\\")\\r\\n    started = time.perf_counter()\\r\\n    discovery = discover_inputs(config)\\r\\n    csv_contracts = {\\"status\\": \\"NOT_REQUESTED\\"}\\r\\n    status = discovery[\\"status\\"]\\r\\n    if config.inspect_csv:\\r\\n        csv_contracts = {\\"status\\": \\"BLOCKED_BY_DISCOVERY\\"}\\r\\n        if discovery[\\"selected_root\\"]:\\r\\n            progress = {\\"attempted_files\\": [], \\"parsed_files\\": [], \\"sha256\\": {}}\\r\\n            discovery[\\"csv_read_attempted\\"] = True\\r\\n            discovery[\\"csv_read_completed\\"] = False\\r\\n            discovery[\\"csv_preflight_completed\\"] = False\\r\\n            try:\\r\\n                csv_contracts = _csv_contracts(discovery[\\"selected_root\\"], config, progress)\\r\\n                status = \\"CONTRACTS_VALID\\" if csv_contracts[\\"status\\"] == \\"VALID\\" else \\"CONTRACTS_INVALID\\"\\r\\n                discovery[\\"csv_preflight_completed\\"] = True\\r\\n            except (InputError, ContractError, OSError) as error:\\r\\n                # Exceptions contain only static contract descriptions; no raw reports.\\r\\n                csv_contracts = {\\"status\\": \\"INVALID\\", \\"error_type\\": type(error).__name__,\\r\\n                                 \\"reason\\": str(error) if isinstance(error, (InputError, ContractError)) else \\"Filesystem read failed\\"}\\r\\n                status = \\"CONTRACTS_INVALID\\"\\r\\n            discovery[\\"csv_read_completed\\"] = len(progress[\\"parsed_files\\"]) == len(MANIFEST_NAMES)\\r\\n            # A parse failure may still have consumed bytes. None means unknown,\\r\\n            # while successful parsed files establish that contents were read.\\r\\n            discovery[\\"csv_contents_read\\"] = True if progress[\\"parsed_files\\"] else None\\r\\n            discovery[\\"csv_read_progress\\"] = progress\\r\\n    return {\\r\\n        \\"schema\\": \\"rsna-knee-preflight-v1\\", \\"phase\\": \\"INITIAL_SCAFFOLD\\", \\"status\\": status,\\r\\n        \\"created_utc\\": datetime.now(timezone.utc).isoformat(), \\"configuration\\": config.to_dict(),\\r\\n        \\"source_identity\\": source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"},\\r\\n        \\"runtime\\": {\\"python\\": platform.python_version(), \\"platform\\": platform.system(), \\"gpu_used\\": False},\\r\\n        \\"discovery\\": discovery, \\"csv_contracts\\": csv_contracts,\\r\\n        \\"phase_b_status\\": \\"NOT_RUN\\", \\"training_ready\\": False, \\"phase1_pass\\": False,\\r\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"observed_study_count\\": None},\\r\\n        \\"model_metrics\\": None,\\r\\n        \\"unassessed\\": [\\"patient grouping\\", \\"report quality and languages\\", \\"DICOM UID inventory\\",\\r\\n                       \\"physical geometry\\", \\"transfer syntaxes and decoding\\", \\"gold provenance\\",\\r\\n                       \\"validation split viability\\", \\"Kaggle account access\\", \\"model and inference runtime\\"],\\r\\n        \\"duration_before_report_write_seconds\\": time.perf_counter() - started,\\r\\n    }\\r\\n\\r\\n\\r\\ndef write_report(report: dict, config: AuditConfig) -> Path:\\r\\n    output = safe_absolute(config.output_dir)\\r\\n    source = safe_absolute(config.input_base)\\r\\n    if output.is_relative_to(source):\\r\\n        raise InputError(\\"Report destination must be outside input_base\\")\\r\\n    payload = json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + \\"\\\\n\\"\\r\\n    if len(payload.encode(\\"utf-8\\")) > config.max_report_bytes:\\r\\n        raise InputError(\\"REPORT_SIZE_LIMIT_EXCEEDED\\")\\r\\n    output.mkdir(parents=True, exist_ok=True)\\r\\n    stamp = datetime.now(timezone.utc).strftime(\\"%Y%m%dT%H%M%S%fZ\\")\\r\\n    path = output / f\\"preflight_{stamp}_{uuid.uuid4().hex[:8]}.json\\"\\r\\n    with path.open(\\"x\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as stream:\\r\\n        stream.write(payload)\\r\\n    return path\\r\\n","cli":"\\"\\"\\"Local entry point with no authenticated services or training commands.\\"\\"\\"\\r\\n\\r\\nimport argparse\\r\\nimport hashlib\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\nimport sys\\r\\n\\r\\nfrom .audit import run_preflight, write_report\\r\\nfrom .config import AuditConfig, ConfigError, load_config\\r\\nfrom .inputs import InputError\\r\\n\\r\\n\\r\\ndef _source_identity() -> dict:\\r\\n    package = Path(__file__).parent\\r\\n    names = (\\"__init__\\", \\"config\\", \\"inputs\\", \\"contracts\\", \\"metrics\\", \\"submission\\", \\"audit\\", \\"cli\\")\\r\\n    return {\\"modules\\": {name: hashlib.sha256((package / (name + \\".py\\")).read_bytes()).hexdigest() for name in names}}\\r\\n\\r\\n\\r\\ndef main(argv=None) -> int:\\r\\n    parser = argparse.ArgumentParser(description=\\"Read-only RSNA input/CSV preflight; no Phase B execution or training\\")\\r\\n    parser.add_argument(\\"--config\\", type=Path)\\r\\n    parser.add_argument(\\"--input-base\\")\\r\\n    parser.add_argument(\\"--data-root\\")\\r\\n    parser.add_argument(\\"--output-dir\\")\\r\\n    parser.add_argument(\\"--inspect-csv\\", action=\\"store_true\\", default=None,\\r\\n                        help=\\"Check bounded CSV contracts; source data is not modified\\")\\r\\n    args = parser.parse_args(argv)\\r\\n    try:\\r\\n        config = load_config(args.config) if args.config else AuditConfig()\\r\\n        overrides = {name: getattr(args, name) for name in (\\"input_base\\", \\"data_root\\", \\"output_dir\\", \\"inspect_csv\\") if getattr(args, name) is not None}\\r\\n        config = config.with_overrides(**overrides)\\r\\n        report = run_preflight(config, source_identity=_source_identity())\\r\\n        path = write_report(report, config)\\r\\n    except (ConfigError, InputError, OSError, ValueError) as error:\\r\\n        message = str(error) if isinstance(error, (ConfigError, InputError)) else \\"Preflight failed; check configuration and filesystem permissions\\"\\r\\n        print(json.dumps({\\"status\\": \\"ERROR\\", \\"error_type\\": type(error).__name__, \\"reason\\": message}), file=sys.stderr)\\r\\n        return 2\\r\\n    print(json.dumps({\\"status\\": report[\\"status\\"], \\"report_path\\": str(path),\\r\\n                      \\"phase_b_status\\": report[\\"phase_b_status\\"], \\"training_ready\\": report[\\"training_ready\\"]}, indent=2))\\r\\n    return 0 if report[\\"status\\"] in (\\"MANIFESTS_LOCATED\\", \\"CONTRACTS_VALID\\") else 2\\r\\n","config":"\\"\\"\\"Small immutable configuration; no credential or environment discovery.\\"\\"\\"\\r\\n\\r\\nfrom dataclasses import asdict, dataclass, fields, replace\\r\\nimport json\\r\\nimport os\\r\\nfrom pathlib import Path\\r\\nimport stat\\r\\nfrom typing import Mapping\\r\\n\\r\\n\\r\\nclass ConfigError(ValueError):\\r\\n    \\"\\"\\"Configuration cannot safely represent the requested preflight.\\"\\"\\"\\r\\n\\r\\n\\r\\n@dataclass(frozen=True)\\r\\nclass AuditConfig:\\r\\n    input_base: str = \\"data\\"\\r\\n    output_dir: str = \\"artifacts\\"\\r\\n    data_root: str | None = None\\r\\n    inspect_csv: bool = False\\r\\n    max_depth: int = 2\\r\\n    max_directories: int = 128\\r\\n    max_entries: int = 2048\\r\\n    max_file_bytes: int = 8 * 1024 * 1024\\r\\n    max_csv_rows: int = 100000\\r\\n    max_report_bytes: int = 1024 * 1024\\r\\n\\r\\n    def __post_init__(self):\\r\\n        for name in (\\"input_base\\", \\"output_dir\\", \\"data_root\\"):\\r\\n            value = getattr(self, name)\\r\\n            if value is None and name == \\"data_root\\":\\r\\n                continue\\r\\n            if not isinstance(value, str) or not value.strip() or value != value.strip():\\r\\n                raise ConfigError(f\\"{name} must be a nonempty unpadded path string\\")\\r\\n            if any(ord(character) < 32 for character in value):\\r\\n                raise ConfigError(f\\"{name} contains a control character\\")\\r\\n        if type(self.inspect_csv) is not bool:\\r\\n            raise ConfigError(\\"inspect_csv must be a JSON boolean\\")\\r\\n        limits = {\\r\\n            \\"max_depth\\": (0, 4), \\"max_directories\\": (1, 1024),\\r\\n            \\"max_entries\\": (1, 65536), \\"max_file_bytes\\": (1, 64 * 1024 * 1024),\\r\\n            \\"max_csv_rows\\": (1, 1000000), \\"max_report_bytes\\": (1, 4 * 1024 * 1024),\\r\\n        }\\r\\n        for name, (lower, upper) in limits.items():\\r\\n            value = getattr(self, name)\\r\\n            if type(value) is not int or not lower <= value <= upper:\\r\\n                raise ConfigError(f\\"{name} must be an integer in [{lower}, {upper}]\\")\\r\\n\\r\\n    @classmethod\\r\\n    def from_dict(cls, values: Mapping):\\r\\n        if not isinstance(values, Mapping):\\r\\n            raise ConfigError(\\"Configuration must be a JSON object\\")\\r\\n        unknown = set(values) - {field.name for field in fields(cls)}\\r\\n        if unknown:\\r\\n            raise ConfigError(\\"Unknown configuration fields: \\" + \\", \\".join(sorted(str(key) for key in unknown)))\\r\\n        return cls(**dict(values))\\r\\n\\r\\n    def to_dict(self) -> dict:\\r\\n        return asdict(self)\\r\\n\\r\\n    def with_overrides(self, **values):\\r\\n        try:\\r\\n            return replace(self, **values)\\r\\n        except TypeError as error:\\r\\n            raise ConfigError(\\"Invalid configuration override\\") from error\\r\\n\\r\\n\\r\\ndef _read_config_values(path: str | Path) -> dict:\\r\\n    \\"\\"\\"Shared bounded JSON reader; never discover environment or credentials.\\"\\"\\"\\r\\n    def unique_pairs(pairs):\\r\\n        values = {}\\r\\n        for key, value in pairs:\\r\\n            if key in values:\\r\\n                raise ConfigError(f\\"Duplicate configuration field: {key}\\")\\r\\n            values[key] = value\\r\\n        return values\\r\\n\\r\\n    source = Path(os.path.abspath(path))\\r\\n    protected = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\r\\n    if source.suffix.casefold() != \\".json\\" or any(part.casefold() in protected or part.casefold().startswith(\\".env\\") for part in source.parts):\\r\\n        raise ConfigError(\\"Configuration must be a project JSON file outside protected paths\\")\\r\\n    for ancestor in (source, *source.parents):\\r\\n        try:\\r\\n            info = ancestor.lstat()\\r\\n        except FileNotFoundError:\\r\\n            continue\\r\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n            raise ConfigError(\\"Configuration cannot use symbolic links or reparse paths\\")\\r\\n    if not stat.S_ISREG(source.lstat().st_mode):\\r\\n        raise ConfigError(\\"Configuration must be a regular project JSON file\\")\\r\\n    with source.open(\\"rb\\") as stream:\\r\\n        raw = stream.read(64 * 1024 + 1)\\r\\n    if len(raw) > 64 * 1024:\\r\\n        raise ConfigError(\\"Configuration exceeds the 64 KiB budget\\")\\r\\n    try:\\r\\n        values = json.loads(raw.decode(\\"utf-8-sig\\"), object_pairs_hook=unique_pairs)\\r\\n    except (json.JSONDecodeError, UnicodeError) as error:\\r\\n        raise ConfigError(\\"Configuration must be valid UTF-8 JSON\\") from error\\r\\n    return values\\r\\n\\r\\n\\r\\ndef load_config(path: str | Path) -> AuditConfig:\\r\\n    return AuditConfig.from_dict(_read_config_values(path))\\r\\n","contracts":"\\"\\"\\"Small, explicit contracts for competition identifiers and supervision.\\r\\n\\r\\nUnknown labels never become negatives. These functions inspect tabular values;\\r\\nthey neither decode images nor infer labels from reports.\\r\\n\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport math\\r\\nimport unicodedata\\r\\nfrom collections import Counter\\r\\nfrom collections.abc import Mapping\\r\\nfrom decimal import Decimal\\r\\nfrom numbers import Real\\r\\nfrom typing import Any\\r\\n\\r\\nTARGETS = (\\r\\n    \\"ACL\\", \\"MCL\\", \\"Medial Meniscus\\", \\"Lateral Meniscus\\", \\"Medial OA\\",\\r\\n    \\"Lateral OA\\", \\"PF OA\\", \\"Effusion\\", \\"Synovitis\\", \\"Baker\'s\\",\\r\\n    \\"Contusion\\", \\"Fracture\\",\\r\\n)\\r\\nSTUDY_ID = \\"StudyInstanceUID\\"\\r\\nSERIES_ID = \\"SeriesInstanceUID\\"\\r\\nSERIES_COLUMNS = (\\r\\n    STUDY_ID, SERIES_ID, \\"Anatomical_Plane\\", \\"Fluid_Sensitive\\", \\"Fat_Suppression\\",\\r\\n)\\r\\n\\r\\n\\r\\nclass ContractError(ValueError):\\r\\n    \\"\\"\\"A table violates a declared data contract.\\"\\"\\"\\r\\n\\r\\n\\r\\ndef validate_identifier(value: str) -> str:\\r\\n    \\"\\"\\"Return an unchanged safe identifier without imposing a numeric UID format.\\"\\"\\"\\r\\n    if not isinstance(value, str) or not value or value != value.strip():\\r\\n        raise ContractError(\\"identifier must be a nonempty, unpadded string\\")\\r\\n    if value in {\\".\\", \\"..\\"} or any(char in value for char in \\"/\\\\\\\\:\\"):\\r\\n        raise ContractError(\\"identifier must be a single safe path component\\")\\r\\n    if any(unicodedata.category(char) in {\\"Cc\\", \\"Cf\\"} for char in value):\\r\\n        raise ContractError(\\"identifier contains a control character\\")\\r\\n    return value\\r\\n\\r\\n\\r\\ndef _numeric(value: Any) -> bool:\\r\\n    return not isinstance(value, bool) and isinstance(value, (Real, Decimal))\\r\\n\\r\\n\\r\\ndef parse_label(value: Any) -> tuple[float | None, str]:\\r\\n    \\"\\"\\"Classify an observed binary label, an unknown cell, or an invalid cell.\\r\\n\\r\\n    Accepted text is deliberately limited to 0, 1, 0.0 and 1.0. A text token\\r\\n    such as \'NaN\' is invalid; an actual floating NaN represents a missing cell.\\r\\n    \\"\\"\\"\\r\\n    if value is None:\\r\\n        return None, \\"unknown\\"\\r\\n    if isinstance(value, str):\\r\\n        token = value.strip()\\r\\n        if not token:\\r\\n            return None, \\"unknown\\"\\r\\n        if token in {\\"0\\", \\"0.0\\", \\"1\\", \\"1.0\\"}:\\r\\n            return float(token), \\"observed\\"\\r\\n        return None, \\"invalid\\"\\r\\n    if isinstance(value, Real) and not isinstance(value, bool):\\r\\n        try:\\r\\n            if math.isnan(float(value)):\\r\\n                return None, \\"unknown\\"\\r\\n        except (OverflowError, ValueError):\\r\\n            return None, \\"invalid\\"\\r\\n    if _numeric(value):\\r\\n        try:\\r\\n            numeric = float(value)\\r\\n        except (OverflowError, ValueError):\\r\\n            return None, \\"invalid\\"\\r\\n        if math.isfinite(numeric) and numeric in (0.0, 1.0):\\r\\n            return numeric, \\"observed\\"\\r\\n    return None, \\"invalid\\"\\r\\n\\r\\n\\r\\ndef _required(row: Mapping[str, Any], required: tuple[str, ...], index: int) -> None:\\r\\n    if not isinstance(row, Mapping):\\r\\n        raise ContractError(f\\"row {index} must be a mapping\\")\\r\\n    missing = [key for key in required if key not in row]\\r\\n    if missing:\\r\\n        raise ContractError(f\\"row {index} is missing required columns: {\', \'.join(missing)}\\")\\r\\n\\r\\n\\r\\ndef label_vector(row: Mapping[str, Any]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Return twelve ordered values and an explicit observation mask.\\r\\n\\r\\n    Unknown and invalid cells retain None; the state list records the\\r\\n    distinction. Only verified binary observations receive a True mask.\\r\\n    \\"\\"\\"\\r\\n    _required(row, TARGETS, 0)\\r\\n    parsed = [parse_label(row[target]) for target in TARGETS]\\r\\n    return {\\r\\n        \\"targets\\": list(TARGETS),\\r\\n        \\"values\\": [value for value, _ in parsed],\\r\\n        \\"observed_mask\\": [state == \\"observed\\" for _, state in parsed],\\r\\n        \\"states\\": [state for _, state in parsed],\\r\\n    }\\r\\n\\r\\n\\r\\ndef audit_studies(rows: list[dict[str, Any]], split: str) -> dict[str, Any]:\\r\\n    \\"\\"\\"Audit study identities and, for training, labels and report coverage.\\"\\"\\"\\r\\n    if split not in {\\"train\\", \\"test\\"}:\\r\\n        raise ContractError(\\"split must be train or test\\")\\r\\n    if not rows:\\r\\n        raise ContractError(\\"study table must contain at least one row\\")\\r\\n    required = (STUDY_ID, *TARGETS, \\"Report\\") if split == \\"train\\" else (STUDY_ID,)\\r\\n    study_ids: list[str] = []\\r\\n    seen: set[str] = set()\\r\\n    statistics = {\\r\\n        target: {\\"observed\\": 0, \\"positive\\": 0, \\"negative\\": 0, \\"unknown\\": 0, \\"invalid\\": 0}\\r\\n        for target in TARGETS\\r\\n    }\\r\\n    label_states: list[dict[str, tuple[float | None, str]]] = []\\r\\n    report_summary = {\\"present\\": 0, \\"empty_or_missing\\": 0, \\"invalid\\": 0}\\r\\n    for index, row in enumerate(rows):\\r\\n        _required(row, required, index)\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in seen:\\r\\n            raise ContractError(f\\"duplicate study identifier at row {index}\\")\\r\\n        seen.add(uid)\\r\\n        study_ids.append(uid)\\r\\n        if split == \\"test\\":\\r\\n            continue\\r\\n        states = {target: parse_label(row[target]) for target in TARGETS}\\r\\n        label_states.append(states)\\r\\n        for target, (label, state) in states.items():\\r\\n            statistics[target][state] += 1\\r\\n            if state == \\"observed\\":\\r\\n                statistics[target][\\"positive\\" if label == 1.0 else \\"negative\\"] += 1\\r\\n        report = row[\\"Report\\"]\\r\\n        if report is None or (isinstance(report, str) and not report.strip()):\\r\\n            report_summary[\\"empty_or_missing\\"] += 1\\r\\n        elif isinstance(report, str):\\r\\n            report_summary[\\"present\\"] += 1\\r\\n        else:\\r\\n            report_summary[\\"invalid\\"] += 1\\r\\n    result: dict[str, Any] = {\\r\\n        \\"status\\": \\"PASS\\", \\"split\\": split, \\"row_count\\": len(rows),\\r\\n        \\"unique_study_count\\": len(seen), \\"study_ids\\": study_ids,\\r\\n    }\\r\\n    if split == \\"test\\":\\r\\n        return result\\r\\n    for stats in statistics.values():\\r\\n        observed = stats[\\"observed\\"]\\r\\n        stats[\\"prevalence\\"] = stats[\\"positive\\"] / observed if observed else None\\r\\n        stats[\\"observation_coverage\\"] = observed / len(rows)\\r\\n    cooccurrence = []\\r\\n    for first_index, first in enumerate(TARGETS):\\r\\n        for second in TARGETS[first_index + 1:]:\\r\\n            jointly_observed = both_positive = 0\\r\\n            for states in label_states:\\r\\n                first_label, first_state = states[first]\\r\\n                second_label, second_state = states[second]\\r\\n                if first_state == second_state == \\"observed\\":\\r\\n                    jointly_observed += 1\\r\\n                    both_positive += first_label == second_label == 1.0\\r\\n            cooccurrence.append({\\r\\n                \\"targets\\": [first, second], \\"jointly_observed\\": jointly_observed,\\r\\n                \\"both_positive\\": both_positive,\\r\\n                \\"both_positive_fraction\\": both_positive / jointly_observed if jointly_observed else None,\\r\\n            })\\r\\n    result.update({\\r\\n        \\"target_statistics\\": statistics, \\"report_summary\\": report_summary,\\r\\n        \\"observation_coverage\\": sum(stats[\\"observed\\"] for stats in statistics.values()) / (len(rows) * len(TARGETS)),\\r\\n        \\"jointly_observed_cooccurrence\\": cooccurrence,\\r\\n    })\\r\\n    if any(stats[\\"invalid\\"] for stats in statistics.values()) or report_summary[\\"invalid\\"]:\\r\\n        result[\\"status\\"] = \\"INVALID\\"\\r\\n    return result\\r\\n\\r\\n\\r\\ndef _metadata_state(value: Any) -> str:\\r\\n    if value is None or (isinstance(value, str) and not value.strip()):\\r\\n        return \\"unknown\\"\\r\\n    if isinstance(value, float):\\r\\n        if math.isnan(float(value)):\\r\\n            return \\"unknown\\"\\r\\n        if not math.isfinite(float(value)):\\r\\n            return \\"invalid\\"\\r\\n    if isinstance(value, (str, bool, int, float)):\\r\\n        return \\"observed\\"\\r\\n    return \\"invalid\\"\\r\\n\\r\\n\\r\\ndef audit_series(rows: list[dict[str, Any]], study_ids: set[str]) -> dict[str, Any]:\\r\\n    \\"\\"\\"Check series uniqueness, study references and exact metadata coverage.\\r\\n\\r\\n    Category vocabularies must be established from real data. No plane or flag\\r\\n    value is coerced here; typed value counts retain the observed representation.\\r\\n    \\"\\"\\"\\r\\n    if not rows:\\r\\n        raise ContractError(\\"series table must contain at least one row\\")\\r\\n    validated_studies = {validate_identifier(uid) for uid in study_ids}\\r\\n    seen: set[str] = set()\\r\\n    referenced: set[str] = set()\\r\\n    metadata_fields = SERIES_COLUMNS[2:]\\r\\n    statistics = {field: {\\"observed\\": 0, \\"unknown\\": 0, \\"invalid\\": 0} for field in metadata_fields}\\r\\n    counts: dict[str, Counter[tuple[str, Any]]] = {field: Counter() for field in metadata_fields}\\r\\n    series_ids: list[str] = []\\r\\n    for index, row in enumerate(rows):\\r\\n        _required(row, SERIES_COLUMNS, index)\\r\\n        study = validate_identifier(row[STUDY_ID])\\r\\n        series = validate_identifier(row[SERIES_ID])\\r\\n        if study not in validated_studies:\\r\\n            raise ContractError(f\\"series row {index} references an absent study\\")\\r\\n        if series in seen:\\r\\n            raise ContractError(f\\"duplicate series identifier at row {index}\\")\\r\\n        seen.add(series)\\r\\n        series_ids.append(series)\\r\\n        referenced.add(study)\\r\\n        for field in metadata_fields:\\r\\n            value = row[field]\\r\\n            state = _metadata_state(value)\\r\\n            statistics[field][state] += 1\\r\\n            if state == \\"observed\\":\\r\\n                counts[field][(type(value).__name__, value)] += 1\\r\\n    for field in metadata_fields:\\r\\n        statistics[field][\\"value_counts\\"] = [\\r\\n            {\\"type\\": value_type, \\"value\\": value, \\"count\\": count}\\r\\n            for (value_type, value), count in sorted(counts[field].items(), key=lambda item: (item[0][0], str(item[0][1])))\\r\\n        ]\\r\\n    return {\\r\\n        \\"status\\": \\"INVALID\\" if any(stats[\\"invalid\\"] for stats in statistics.values()) else \\"PASS\\",\\r\\n        \\"row_count\\": len(rows), \\"unique_series_count\\": len(seen),\\r\\n        \\"study_count_with_series\\": len(referenced), \\"study_count_without_series\\": len(validated_studies - referenced),\\r\\n        \\"series_ids\\": series_ids, \\"metadata_statistics\\": statistics,\\r\\n    }\\r\\n","dicom_audit":"\\"\\"\\"Bounded DICOM smoke audit; sampled aggregates only, never image caches or IDs.\\n\\nPydicom is optional and is imported only when this audit is requested. Time\\nlimits are cooperative: stream operations and attempt boundaries check elapsed\\ntime, but an already running native decoder cannot be forcibly interrupted.\\n\\"\\"\\"\\n\\nfrom collections import Counter, defaultdict\\nimport importlib\\nfrom itertools import islice\\nimport logging\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport stat\\nimport statistics\\nimport struct\\nimport time\\nimport warnings\\n\\nfrom .inputs import InputError, safe_absolute\\n\\n\\n_UID = re.compile(r\\"(?:0|[1-9][0-9]*)(?:\\\\.(?:0|[1-9][0-9]*))*\\\\Z\\")\\n_MODALITIES = frozenset(\\"MR CT CR DX US NM PT XA RF MG OT SR SC SEG RTIMAGE RTDOSE RTSTRUCT\\".split())\\n_PHOTOMETRIC = frozenset((\\"MONOCHROME1\\", \\"MONOCHROME2\\", \\"RGB\\", \\"PALETTE COLOR\\", \\"YBR_FULL\\", \\"YBR_FULL_422\\", \\"YBR_PARTIAL_422\\", \\"YBR_PARTIAL_420\\", \\"YBR_ICT\\", \\"YBR_RCT\\"))\\n_IMAGE_TYPES = frozenset(\\"ORIGINAL DERIVED PRIMARY SECONDARY AXIAL SAGITTAL CORONAL LOCALIZER OTHER NONE MPR PROJECTION VOLUME MAXIMUM MINIMUM AVERAGE MAGNITUDE PHASE REAL IMAGINARY\\".split())\\n_PLUGINS = frozenset((\\"numpy\\", \\"pydicom\\", \\"pylibjpeg\\", \\"pillow\\", \\"gdcm\\"))\\n\\n\\nclass _BudgetExceeded(Exception):\\n    pass\\n\\n\\nclass _TimeExceeded(Exception):\\n    pass\\n\\n\\nclass _UnsupportedEncoding(Exception):\\n    pass\\n\\n\\nclass _BoundedStream:\\n    \\"\\"\\"Cap cumulative reads and accessible offsets, including parser seeks.\\"\\"\\"\\n\\n    def __init__(self, stream, limit, deadline):\\n        self.stream = stream\\n        self.limit = limit\\n        self.deadline = deadline\\n        self.bytes_read = 0\\n        self.size = os.fstat(stream.fileno()).st_size\\n\\n    def _check(self):\\n        if time.perf_counter() >= self.deadline:\\n            raise _TimeExceeded()\\n\\n    def read(self, size=-1):\\n        self._check()\\n        remaining = self.limit - self.bytes_read\\n        requested = remaining if size is None or size < 0 else size\\n        if requested > remaining:\\n            raise _BudgetExceeded()\\n        if requested == 0 and (size is None or size < 0) and self.stream.tell() < self.size:\\n            raise _BudgetExceeded()\\n        data = self.stream.read(requested)\\n        self.bytes_read += len(data)\\n        return data\\n\\n    def readinto(self, buffer):\\n        data = self.read(len(buffer))\\n        buffer[:len(data)] = data\\n        return len(data)\\n\\n    def seek(self, offset, whence=0):\\n        self._check()\\n        target = offset if whence == 0 else self.stream.tell() + offset if whence == 1 else self.size + offset if whence == 2 else -1\\n        if target < 0 or target > self.limit:\\n            raise _BudgetExceeded()\\n        return self.stream.seek(target)\\n\\n    def tell(self):\\n        return self.stream.tell()\\n\\n\\ndef _load_pydicom():\\n    return importlib.import_module(\\"pydicom\\")\\n\\n\\ndef _reject_deflated_dataset(stream):\\n    \\"\\"\\"Inspect bounded Part10 meta bytes before pydicom can inflate a dataset.\\n\\n    Group 0002 uses explicit VR little endian even when the following dataset\\n    uses another syntax. No metadata text is returned or logged.\\n    \\"\\"\\"\\n    prefix = stream.read(132)\\n    if len(prefix) == 132 and prefix[128:] == b\\"DICM\\":\\n        long_vrs = {b\\"OB\\", b\\"OD\\", b\\"OF\\", b\\"OL\\", b\\"OV\\", b\\"OW\\", b\\"SQ\\", b\\"SV\\", b\\"UC\\", b\\"UN\\", b\\"UR\\", b\\"UT\\", b\\"UV\\"}\\n        for _ in range(256):\\n            header = stream.read(8)\\n            if len(header) != 8:\\n                break\\n            group, element = struct.unpack(\\"<HH\\", header[:4])\\n            if group != 2:\\n                break\\n            vr = header[4:6]\\n            if vr in long_vrs:\\n                length_bytes = stream.read(4)\\n                if len(length_bytes) != 4:\\n                    raise ValueError(\\"Invalid file meta\\")\\n                length = struct.unpack(\\"<I\\", length_bytes)[0]\\n            else:\\n                length = struct.unpack(\\"<H\\", header[6:8])[0]\\n            if element == 0x10:\\n                if length > 66:\\n                    raise ValueError(\\"Invalid transfer syntax metadata\\")\\n                syntax = stream.read(length).strip(b\\"\\\\0 \\\\t\\\\n\\\\r\\\\v\\\\f\\")\\n                if syntax == b\\"1.2.840.10008.1.2.1.99\\":\\n                    raise _UnsupportedEncoding()\\n            else:\\n                stream.seek(length, 1)\\n        else:\\n            raise _BudgetExceeded()\\n    stream.seek(0)\\n\\n\\ndef _value(dataset, name):\\n    try:\\n        return getattr(dataset, name, None)\\n    except Exception:\\n        return None\\n\\n\\ndef _text(value, limit=256):\\n    if value is None:\\n        return None\\n    try:\\n        text = str(value)\\n    except Exception:\\n        return None\\n    return text if text and len(text) <= limit and not any(ord(c) < 32 for c in text) else None\\n\\n\\ndef _uid(value):\\n    text = _text(value, 64)\\n    return text if text is not None and _UID.fullmatch(text) else None\\n\\n\\ndef _number(value, lower=-1000000, upper=1000000):\\n    if isinstance(value, bool) or value is None:\\n        return None\\n    try:\\n        number = float(value)\\n    except (TypeError, ValueError, OverflowError):\\n        return None\\n    return number if math.isfinite(number) and lower <= number <= upper else None\\n\\n\\ndef _integer(value, lower=0, upper=1000000):\\n    number = _number(value, lower, upper)\\n    return int(number) if number is not None and number.is_integer() else None\\n\\n\\ndef _vector(value, length, lower=-1000000, upper=1000000):\\n    if isinstance(value, (str, bytes)):\\n        return None\\n    try:\\n        if len(value) != length:\\n            return None\\n        numbers = tuple(_number(item, lower, upper) for item in value)\\n    except (TypeError, ValueError):\\n        return None\\n    return numbers if all(item is not None for item in numbers) else None\\n\\n\\ndef _statistics(values):\\n    return {\\"count\\": len(values), \\"minimum\\": min(values) if values else None,\\n            \\"maximum\\": max(values) if values else None,\\n            \\"median\\": statistics.median(values) if values else None}\\n\\n\\ndef _histogram(counter, limit=24):\\n    items = sorted(counter.items(), key=lambda item: (-item[1], item[0]))\\n    return {\\"values\\": {key: value for key, value in items[:limit]},\\n            \\"other_count\\": sum(value for _, value in items[limit:]),\\n            \\"distinct_count\\": len(items), \\"maximum_categories\\": limit}\\n\\n\\ndef _normal(orientation):\\n    if orientation is None:\\n        return None\\n    row, column = orientation[:3], orientation[3:]\\n    if any(abs(sum(component * component for component in vector) - 1) > 0.05 for vector in (row, column)):\\n        return None\\n    if abs(sum(a * b for a, b in zip(row, column))) > 0.05:\\n        return None\\n    cross = (row[1] * column[2] - row[2] * column[1], row[2] * column[0] - row[0] * column[2], row[0] * column[1] - row[1] * column[0])\\n    norm = math.sqrt(sum(component * component for component in cross))\\n    return tuple(component / norm for component in cross) if norm > 0 else None\\n\\n\\ndef _geometry(groups):\\n    gaps = []\\n    result = {\\"status\\": \\"NOT_ASSESSED\\", \\"sampled_series_count\\": len(groups),\\n              \\"physical_order_supported_series_count\\": 0, \\"orientation_inconsistent_series_count\\": 0,\\n              \\"insufficient_position_series_count\\": 0, \\"duplicate_position_count\\": 0,\\n              \\"duplicate_instance_number_count\\": 0, \\"instance_order_direction_change_series_count\\": 0,\\n              \\"large_sampled_gap_count\\": 0,\\n              \\"method\\": \\"Project ImagePositionPatient onto cross(row, column) from ImageOrientationPatient; sort projections. SliceThickness is never substituted for position spacing.\\",\\n              \\"scope\\": \\"Sampled positions only. Large sample gaps do not establish missing slices or complete-series spacing.\\"}\\n    for samples in groups.values():\\n        usable = [(normal, position, instance) for normal, position, instance in samples if normal is not None and position is not None]\\n        instances = [instance for _, _, instance in samples if instance is not None]\\n        result[\\"duplicate_instance_number_count\\"] += len(instances) - len(set(instances))\\n        if len(usable) < 2:\\n            result[\\"insufficient_position_series_count\\"] += 1\\n            continue\\n        reference = usable[0][0]\\n        if any(sum(a * b for a, b in zip(reference, normal)) < 0.999 for normal, _, _ in usable):\\n            result[\\"orientation_inconsistent_series_count\\"] += 1\\n            continue\\n        positions = sorted(sum(a * b for a, b in zip(reference, position)) for _, position, _ in usable)\\n        distances = [right - left for left, right in zip(positions, positions[1:])]\\n        result[\\"duplicate_position_count\\"] += sum(gap <= 0.0001 for gap in distances)\\n        positive = [gap for gap in distances if gap > 0.0001]\\n        gaps.extend(positive)\\n        if positive:\\n            median = statistics.median(positive)\\n            result[\\"large_sampled_gap_count\\"] += sum(gap > median * 1.5 for gap in positive)\\n        by_instance = sorted((instance, sum(a * b for a, b in zip(reference, position))) for _, position, instance in usable if instance is not None)\\n        differences = [right[1] - left[1] for left, right in zip(by_instance, by_instance[1:])]\\n        if any(gap > 0.0001 for gap in differences) and any(gap < -0.0001 for gap in differences):\\n            result[\\"instance_order_direction_change_series_count\\"] += 1\\n        result[\\"physical_order_supported_series_count\\"] += 1\\n    result[\\"sampled_position_gap_statistics_mm\\"] = _statistics(gaps)\\n    if result[\\"physical_order_supported_series_count\\"]:\\n        result[\\"status\\"] = \\"PASS\\"\\n    if result[\\"orientation_inconsistent_series_count\\"] or result[\\"duplicate_position_count\\"] or result[\\"duplicate_instance_number_count\\"] or result[\\"instance_order_direction_change_series_count\\"]:\\n        result[\\"status\\"] = \\"FAIL\\"\\n    return result\\n\\n\\ndef _codec_availability(pydicom, syntax):\\n    result = {\\"availability_status\\": \\"NOT_ASSESSED\\", \\"available_plugins\\": [], \\"sample_decode_status\\": \\"NOT_ASSESSED\\", \\"decode_attempts\\": 0, \\"decode_successes\\": 0, \\"decode_failures\\": 0}\\n    try:\\n        pixels = getattr(pydicom, \\"pixels\\", None)\\n        get_decoder = getattr(pixels, \\"get_decoder\\", None)\\n        if get_decoder:\\n            decoder = get_decoder(syntax)\\n            available = bool(decoder.is_available)\\n            result.update(availability_status=\\"PASS\\" if available else \\"BLOCKED\\",\\n                          available_plugins=sorted({str(name).casefold() for name in decoder.available_plugins if str(name).casefold() in _PLUGINS}))\\n        else:\\n            handlers = getattr(getattr(pydicom, \\"config\\", None), \\"pixel_data_handlers\\", None)\\n            if handlers is not None:\\n                supporting = [handler for handler in handlers if handler.supports_transfer_syntax(syntax) and handler.is_available()]\\n                result[\\"availability_status\\"] = \\"PASS\\" if supporting else \\"BLOCKED\\"\\n                result[\\"available_plugins\\"] = sorted({str(getattr(handler, \\"HANDLER_NAME\\", \\"\\")).casefold() for handler in supporting} & _PLUGINS)\\n    except Exception:\\n        pass\\n    return result\\n\\n\\ndef _record_blocked_deflate(result):\\n    \\"\\"\\"Record the one policy-blocked numeric syntax without codec assumptions.\\"\\"\\"\\n    syntax = \\"1.2.840.10008.1.2.1.99\\"\\n    observed = result[\\"transfer_syntaxes\\"][\\"observed\\"]\\n    if syntax not in observed:\\n        observed[syntax] = {\\"availability_status\\": \\"NOT_ASSESSED\\", \\"available_plugins\\": [],\\n                            \\"sample_decode_status\\": \\"NOT_ASSESSED\\", \\"decode_attempts\\": 0,\\n                            \\"decode_successes\\": 0, \\"decode_failures\\": 0, \\"file_meta_count\\": 0,\\n                            \\"audit_policy_status\\": \\"BLOCKED\\",\\n                            \\"reason\\": \\"Deflated whole-dataset parsing is not attempted by this bounded audit\\"}\\n    observed[syntax][\\"file_meta_count\\"] += 1\\n\\n\\ndef _failure(error):\\n    if isinstance(error, _TimeExceeded):\\n        return \\"time_limit\\"\\n    if isinstance(error, _BudgetExceeded):\\n        return \\"read_byte_limit\\"\\n    if isinstance(error, FileNotFoundError):\\n        return \\"file_missing\\"\\n    if isinstance(error, PermissionError):\\n        return \\"inaccessible\\"\\n    if isinstance(error, InputError):\\n        return \\"unsafe_path\\"\\n    if isinstance(error, MemoryError):\\n        return \\"memory_limit\\"\\n    if isinstance(error, OSError):\\n        return \\"filesystem_error\\"\\n    if isinstance(error, (ImportError, ModuleNotFoundError)):\\n        return \\"decoder_dependency_unavailable\\"\\n    if isinstance(error, NotImplementedError):\\n        return \\"unsupported_encoding\\"\\n    if isinstance(error, _UnsupportedEncoding):\\n        return \\"deflated_dataset_not_audited\\"\\n    return \\"invalid_header_or_decode\\"\\n\\n\\ndef _checked_path(record, base):\\n    source = safe_absolute(record[\\"path\\"])\\n    series = safe_absolute(record[\\"series_path\\"])\\n    if not source.is_relative_to(base) or not series.is_relative_to(base) or not source.is_relative_to(series):\\n        raise InputError(\\"DICOM path is outside the permitted input series\\")\\n    info = source.lstat()\\n    if not stat.S_ISREG(info.st_mode):\\n        raise InputError(\\"DICOM source is not a regular file\\")\\n    return source, info.st_size\\n\\n\\ndef inspect_dicom(file_records, config) -> dict:\\n    \\"\\"\\"Suppress parser diagnostics, including child loggers created lazily.\\n\\n    The notebook runs this audit sequentially. Its temporary process-wide log\\n    and warning suppression also affects other threads; both states are restored\\n    on exit. Disabling only the parent pydicom logger would not suppress child\\n    loggers with their own handlers or propagated records.\\n    \\"\\"\\"\\n    was_disabled = logging.root.manager.disable\\n    logging.disable(logging.CRITICAL)\\n    try:\\n        with warnings.catch_warnings():\\n            warnings.simplefilter(\\"ignore\\")\\n            return _inspect_dicom(file_records, config)\\n    finally:\\n        logging.disable(was_disabled)\\n\\n\\ndef _inspect_dicom(file_records, config) -> dict:\\n    \\"\\"\\"Inspect inventory-selected files in deterministic order with explicit scope.\\n\\n    Records are the already bounded, stratified inventory sample. No record,\\n    identifier, raw string exception, report or pixel array is serialized.\\n    \\"\\"\\"\\n    started = time.perf_counter()\\n    deadline = started + config.max_dicom_seconds\\n    limits = {name: getattr(config, name) for name in (\\"max_header_reads\\", \\"max_header_bytes\\", \\"max_dicom_file_bytes\\", \\"max_pixel_decodes\\", \\"max_pixel_elements\\", \\"max_dicom_seconds\\")}\\n    result = {\\"status\\": \\"NOT_ASSESSED\\", \\"scope\\": \\"Bounded deterministic inventory sample only; no complete DICOM integrity or patient mapping claim.\\",\\n              \\"limits\\": limits, \\"runtime_seconds\\": 0.0, \\"time_limit_reached\\": False,\\n              \\"time_limit_enforcement\\": \\"Cooperative between reads and attempts; cannot interrupt an already running native decoder.\\",\\n              \\"headers\\": {\\"status\\": \\"NOT_ASSESSED\\", \\"attempts\\": 0, \\"successes\\": 0, \\"failures\\": 0, \\"bytes_read\\": 0, \\"failure_categories\\": {}, \\"sample_limit_reached\\": False},\\n              \\"identifiers\\": {}, \\"metadata\\": {}, \\"geometry\\": {\\"status\\": \\"NOT_ASSESSED\\"},\\n              \\"transfer_syntaxes\\": {\\"observed\\": {}, \\"unencountered_syntax_support\\": \\"NOT_ASSESSED\\"},\\n              \\"decoding\\": {\\"status\\": \\"NOT_ASSESSED\\", \\"requested\\": config.max_pixel_decodes > 0, \\"attempts\\": 0, \\"successes\\": 0, \\"failures\\": 0, \\"skipped_resource_limits\\": 0, \\"failure_categories\\": {}, \\"skip_categories\\": {}, \\"bytes_read\\": 0, \\"array_shapes\\": {}, \\"constant_arrays\\": 0, \\"all_zero_arrays\\": 0},\\n              \\"patient_grouping\\": {\\"status\\": \\"NOT_ASSESSED\\", \\"trustworthy_full_study_mapping\\": False, \\"reason\\": \\"Sampled PatientID availability and consistency do not establish reliable patient grouping for all studies.\\"}}\\n    try:\\n        pydicom = _load_pydicom()\\n    except ImportError:\\n        result.update(status=\\"BLOCKED\\", dependency={\\"pydicom\\": \\"UNAVAILABLE\\", \\"action\\": \\"No installation attempted\\"})\\n        result[\\"headers\\"][\\"status\\"] = result[\\"decoding\\"][\\"status\\"] = \\"BLOCKED\\"\\n        result[\\"runtime_seconds\\"] = time.perf_counter() - started\\n        return result\\n    result[\\"dependency\\"] = {\\"pydicom\\": \\"AVAILABLE\\"}\\n    header_errors, decode_errors, skips = Counter(), Counter(), Counter()\\n    missing, malformed, numeric = Counter(), Counter(), defaultdict(list)\\n    categorical = {name: Counter() for name in (\\"Modality\\", \\"ImageType\\", \\"PhotometricInterpretation\\", \\"dimensions\\", \\"PixelSpacing\\", \\"ImageOrientationPatient\\", \\"BitsStored\\", \\"PixelRepresentation\\")}\\n    presence = Counter()\\n    uid_sets = {name: set() for name in (\\"StudyInstanceUID\\", \\"SeriesInstanceUID\\", \\"SOPInstanceUID\\")}\\n    groups, patient_ids, study_ids, series_ids = defaultdict(list), defaultdict(set), defaultdict(set), defaultdict(set)\\n    id_counts = Counter()\\n    syntax_counts, shapes, splits = Counter(), Counter(), Counter()\\n    decode_candidates = []\\n    try:\\n        base = safe_absolute(config.input_base)\\n    except (InputError, OSError):\\n        result.update(status=\\"BLOCKED\\", reason=\\"INPUT_PATH_UNSAFE\\")\\n        result[\\"runtime_seconds\\"] = time.perf_counter() - started\\n        return result\\n\\n    for number, record in enumerate(file_records):\\n        if number >= config.max_header_reads:\\n            result[\\"headers\\"][\\"sample_limit_reached\\"] = True\\n            break\\n        if time.perf_counter() >= deadline:\\n            result[\\"time_limit_reached\\"] = True\\n            break\\n        result[\\"headers\\"][\\"attempts\\"] += 1\\n        stream = None\\n        try:\\n            source, size = _checked_path(record, base)\\n            with source.open(\\"rb\\") as raw:\\n                stream = _BoundedStream(raw, config.max_header_bytes, deadline)\\n                _reject_deflated_dataset(stream)\\n                with warnings.catch_warnings():\\n                    warnings.simplefilter(\\"ignore\\")\\n                    dataset = pydicom.dcmread(stream, stop_before_pixels=True, force=False)\\n            row = record[\\"row\\"]\\n            key = (record.get(\\"split\\", \\"\\"), row.get(\\"StudyInstanceUID\\", \\"\\"), row.get(\\"SeriesInstanceUID\\", \\"\\"))\\n            identifiers = {}\\n            for name in uid_sets:\\n                value = _value(dataset, name)\\n                identifier = _uid(value)\\n                if value is None or value == \\"\\":\\n                    missing[name] += 1\\n                elif identifier is None:\\n                    malformed[name] += 1\\n                else:\\n                    id_counts[name + \\"_present\\"] += 1\\n                    if name == \\"SOPInstanceUID\\" and identifier in uid_sets[name]:\\n                        id_counts[\\"duplicate_sop_instance_uid_count\\"] += 1\\n                    uid_sets[name].add(identifier)\\n                identifiers[name] = identifier\\n            for name, target in ((\\"StudyInstanceUID\\", study_ids), (\\"SeriesInstanceUID\\", series_ids)):\\n                identifier = identifiers[name]\\n                if identifier is not None:\\n                    target[key].add(identifier)\\n                    if identifier != row.get(name):\\n                        id_counts[name + \\"_manifest_mismatch_count\\"] += 1\\n            patient = _text(_value(dataset, \\"PatientID\\"))\\n            if patient is not None:\\n                id_counts[\\"PatientID_present\\"] += 1\\n                patient_ids[key].add(patient)\\n            else:\\n                missing[\\"PatientID\\"] += 1\\n            if _value(dataset, \\"SeriesDescription\\") not in (None, \\"\\"):\\n                presence[\\"SeriesDescription\\"] += 1\\n            for name, allowed in ((\\"Modality\\", _MODALITIES), (\\"PhotometricInterpretation\\", _PHOTOMETRIC)):\\n                value = _text(_value(dataset, name), 32)\\n                if value is None:\\n                    missing[name] += 1\\n                else:\\n                    categorical[name][value if value in allowed else \\"OTHER\\"] += 1\\n            image_type = _value(dataset, \\"ImageType\\")\\n            if image_type is None:\\n                missing[\\"ImageType\\"] += 1\\n            else:\\n                tokens = image_type.split(\\"\\\\\\\\\\", 8)[:8] if isinstance(image_type, str) else list(islice(iter(image_type), 8))\\n                safe_tokens = [_text(token, 32) for token in tokens[:8]]\\n                categorical[\\"ImageType\\"][\\"\\\\\\\\\\".join(token if token in _IMAGE_TYPES else \\"OTHER\\" for token in safe_tokens)] += 1\\n            integers = {}\\n            for name, lower, upper in ((\\"Rows\\", 1, 1000000), (\\"Columns\\", 1, 1000000), (\\"BitsStored\\", 1, 64), (\\"PixelRepresentation\\", 0, 1), (\\"InstanceNumber\\", -1000000, 1000000)):\\n                raw_value = _value(dataset, name)\\n                value = _integer(raw_value, lower, upper)\\n                integers[name] = value\\n                if raw_value is None:\\n                    missing[name] += 1\\n                elif value is None:\\n                    malformed[name] += 1\\n                else:\\n                    numeric[name].append(value)\\n                    if name in categorical:\\n                        categorical[name][str(value)] += 1\\n            if integers[\\"Rows\\"] and integers[\\"Columns\\"]:\\n                categorical[\\"dimensions\\"][f\\"{integers[\'Rows\']}x{integers[\'Columns\']}\\"] += 1\\n            spacing = _vector(_value(dataset, \\"PixelSpacing\\"), 2, 0.000001)\\n            orientation = _vector(_value(dataset, \\"ImageOrientationPatient\\"), 6, -1.1, 1.1)\\n            position = _vector(_value(dataset, \\"ImagePositionPatient\\"), 3)\\n            for name, value in ((\\"PixelSpacing\\", spacing), (\\"ImageOrientationPatient\\", orientation), (\\"ImagePositionPatient\\", position)):\\n                if value is None:\\n                    (missing if _value(dataset, name) is None else malformed)[name] += 1\\n                else:\\n                    presence[name] += 1\\n                    if name in categorical:\\n                        categorical[name][\\",\\".join(f\\"{item:.4g}\\" for item in value)] += 1\\n                    for item in value:\\n                        numeric[name + \\"_components\\"].append(item)\\n            for name in (\\"SpacingBetweenSlices\\", \\"SliceThickness\\", \\"RescaleSlope\\", \\"RescaleIntercept\\"):\\n                raw_value = _value(dataset, name)\\n                value = _number(raw_value)\\n                if raw_value is None:\\n                    missing[name] += 1\\n                elif value is None:\\n                    malformed[name] += 1\\n                else:\\n                    numeric[name].append(value)\\n            normal = _normal(orientation)\\n            if orientation is not None and normal is None:\\n                malformed[\\"ImageOrientationPatient_basis\\"] += 1\\n            groups[key].append((normal, position, integers[\\"InstanceNumber\\"]))\\n            syntax_value = _value(_value(dataset, \\"file_meta\\"), \\"TransferSyntaxUID\\")\\n            syntax = _uid(syntax_value)\\n            if syntax is None:\\n                (missing if syntax_value is None else malformed)[\\"TransferSyntaxUID\\"] += 1\\n            else:\\n                syntax_counts[syntax] += 1\\n                if syntax not in result[\\"transfer_syntaxes\\"][\\"observed\\"]:\\n                    result[\\"transfer_syntaxes\\"][\\"observed\\"][syntax] = _codec_availability(pydicom, syntax)\\n            splits[record.get(\\"split\\") if record.get(\\"split\\") in (\\"train\\", \\"test\\") else \\"other\\"] += 1\\n            result[\\"headers\\"][\\"successes\\"] += 1\\n            decode_candidates.append((record, dataset, syntax, size))\\n        except Exception as error:\\n            category = _failure(error)\\n            if isinstance(error, _UnsupportedEncoding):\\n                _record_blocked_deflate(result)\\n            header_errors[category] += 1\\n            result[\\"headers\\"][\\"failures\\"] += 1\\n            if category == \\"time_limit\\":\\n                result[\\"time_limit_reached\\"] = True\\n        finally:\\n            if stream is not None:\\n                result[\\"headers\\"][\\"bytes_read\\"] += stream.bytes_read\\n        if result[\\"time_limit_reached\\"]:\\n            break\\n\\n    for record, dataset, syntax, _ in decode_candidates:\\n        if result[\\"decoding\\"][\\"attempts\\"] >= config.max_pixel_decodes:\\n            break\\n        if time.perf_counter() >= deadline:\\n            result[\\"time_limit_reached\\"] = True\\n            break\\n        rows = _integer(_value(dataset, \\"Rows\\"), 1)\\n        columns = _integer(_value(dataset, \\"Columns\\"), 1)\\n        frames = _integer(_value(dataset, \\"NumberOfFrames\\") if _value(dataset, \\"NumberOfFrames\\") is not None else 1, 1)\\n        components = _integer(_value(dataset, \\"SamplesPerPixel\\"), 1, 16)\\n        bits = _integer(_value(dataset, \\"BitsAllocated\\"), 1, 64)\\n        stored = _integer(_value(dataset, \\"BitsStored\\"), 1, 64)\\n        if None in (rows, columns, frames, components, bits, stored) or bits not in (1, 8, 16, 32, 64) or stored > bits:\\n            skips[\\"invalid_pixel_metadata\\"] += 1\\n            continue\\n        elements = rows * columns * frames * components\\n        declared_bytes = (elements * bits + 7) // 8\\n        if elements > config.max_pixel_elements or declared_bytes > config.max_dicom_file_bytes:\\n            result[\\"decoding\\"][\\"skipped_resource_limits\\"] += 1\\n            skips[\\"declared_pixel_budget\\"] += 1\\n            continue\\n        stream = None\\n        attempted = False\\n        array = full = None\\n        try:\\n            source, size = _checked_path(record, base)\\n            if size > config.max_dicom_file_bytes:\\n                result[\\"decoding\\"][\\"skipped_resource_limits\\"] += 1\\n                skips[\\"file_byte_budget\\"] += 1\\n                continue\\n            # Count an attempt when the bounded full read begins, including failures.\\n            result[\\"decoding\\"][\\"attempts\\"] += 1\\n            attempted = True\\n            if syntax is not None:\\n                result[\\"transfer_syntaxes\\"][\\"observed\\"][syntax][\\"decode_attempts\\"] += 1\\n            with source.open(\\"rb\\") as raw:\\n                stream = _BoundedStream(raw, config.max_dicom_file_bytes, deadline)\\n                _reject_deflated_dataset(stream)\\n                with warnings.catch_warnings():\\n                    warnings.simplefilter(\\"ignore\\")\\n                    full = pydicom.dcmread(stream, stop_before_pixels=False, force=False)\\n                    # Recheck dimensions before invoking a codec, in case a file\\n                    # changed between its header and full-read smoke test.\\n                    full_frames = _value(full, \\"NumberOfFrames\\")\\n                    full_frames = _integer(1 if full_frames is None else full_frames, 1)\\n                    full_shape = (_integer(_value(full, \\"Rows\\"), 1), _integer(_value(full, \\"Columns\\"), 1), full_frames, _integer(_value(full, \\"SamplesPerPixel\\"), 1, 16), _integer(_value(full, \\"BitsAllocated\\"), 1, 64))\\n                    if full_shape != (rows, columns, frames, components, bits):\\n                        raise _BudgetExceeded()\\n                    array = full.pixel_array\\n            shape = tuple(int(item) for item in array.shape)\\n            expected = ((frames,) if frames > 1 else ()) + (rows, columns) + ((components,) if components > 1 else ())\\n            if shape != expected or int(array.size) != elements or int(array.nbytes) > config.max_dicom_file_bytes or getattr(array.dtype, \\"kind\\", None) not in (\\"i\\", \\"u\\", \\"f\\"):\\n                decode_errors[\\"array_shape_or_type\\"] += 1\\n                result[\\"decoding\\"][\\"failures\\"] += 1\\n            else:\\n                minimum, maximum = float(array.min()), float(array.max())\\n                if not math.isfinite(minimum) or not math.isfinite(maximum):\\n                    decode_errors[\\"nonfinite_pixels\\"] += 1\\n                    result[\\"decoding\\"][\\"failures\\"] += 1\\n                else:\\n                    result[\\"decoding\\"][\\"successes\\"] += 1\\n                    shapes[\\"x\\".join(str(item) for item in shape)] += 1\\n                    result[\\"decoding\\"][\\"constant_arrays\\"] += int(minimum == maximum)\\n                    result[\\"decoding\\"][\\"all_zero_arrays\\"] += int(minimum == maximum == 0)\\n                    if syntax is not None:\\n                        result[\\"transfer_syntaxes\\"][\\"observed\\"][syntax][\\"decode_successes\\"] += 1\\n        except Exception as error:\\n            category = _failure(error)\\n            if isinstance(error, _UnsupportedEncoding):\\n                _record_blocked_deflate(result)\\n            # Path errors before the full read are skips, not decoding attempts.\\n            if not attempted:\\n                skips[category] += 1\\n            else:\\n                decode_errors[category] += 1\\n                result[\\"decoding\\"][\\"failures\\"] += 1\\n            if category == \\"time_limit\\":\\n                result[\\"time_limit_reached\\"] = True\\n        finally:\\n            if stream is not None:\\n                result[\\"decoding\\"][\\"bytes_read\\"] += stream.bytes_read\\n            array = full = None\\n\\n    for name, value in result[\\"transfer_syntaxes\\"][\\"observed\\"].items():\\n        value[\\"header_count\\"] = syntax_counts[name]\\n        value[\\"decode_failures\\"] = value[\\"decode_attempts\\"] - value[\\"decode_successes\\"]\\n        if value[\\"decode_attempts\\"]:\\n            value[\\"sample_decode_status\\"] = \\"FAIL\\" if value[\\"decode_failures\\"] else \\"PASS\\"\\n    id_counts[\\"StudyInstanceUID_inconsistent_series_count\\"] = sum(len(values) > 1 for values in study_ids.values())\\n    id_counts[\\"SeriesInstanceUID_inconsistent_series_count\\"] = sum(len(values) > 1 for values in series_ids.values())\\n    id_counts[\\"PatientID_inconsistent_series_count\\"] = sum(len(values) > 1 for values in patient_ids.values())\\n    identity_fail = any(value for name, value in id_counts.items() if \\"mismatch\\" in name or \\"inconsistent\\" in name or name == \\"duplicate_sop_instance_uid_count\\") or any(missing[name] or malformed[name] for name in uid_sets)\\n    result[\\"identifiers\\"] = {\\"status\\": \\"FAIL\\" if identity_fail else \\"PASS\\" if groups else \\"NOT_ASSESSED\\", \\"counts\\": dict(id_counts), \\"distinct_header_uid_counts\\": {name: len(values) for name, values in uid_sets.items()}, \\"sampled_series_count\\": len(groups), \\"raw_identifiers_emitted\\": False}\\n    result[\\"metadata\\"] = {\\"status\\": \\"FAIL\\" if malformed else \\"PASS\\" if groups else \\"NOT_ASSESSED\\", \\"missing_field_counts\\": dict(missing), \\"malformed_field_counts\\": dict(malformed), \\"presence_counts\\": dict(presence), \\"numeric_statistics\\": {name: _statistics(values) for name, values in sorted(numeric.items())}, \\"distributions\\": {name: _histogram(values) for name, values in categorical.items()}, \\"successful_header_split_counts\\": dict(splits), \\"SeriesDescription_policy\\": \\"Presence only; no description text emitted\\"}\\n    result[\\"geometry\\"] = _geometry(groups)\\n    result[\\"headers\\"][\\"failure_categories\\"] = dict(header_errors)\\n    result[\\"decoding\\"].update(failure_categories=dict(decode_errors), skip_categories=dict(skips), array_shapes=_histogram(shapes))\\n    blocked_categories = {\\"time_limit\\", \\"read_byte_limit\\", \\"memory_limit\\", \\"deflated_dataset_not_audited\\"}\\n    header_bad = any(count for name, count in header_errors.items() if name not in blocked_categories)\\n    result[\\"headers\\"][\\"status\\"] = \\"FAIL\\" if header_bad else \\"BLOCKED\\" if header_errors else \\"PASS\\" if result[\\"headers\\"][\\"successes\\"] else \\"NOT_ASSESSED\\"\\n    decode_blocked_categories = blocked_categories | {\\"decoder_dependency_unavailable\\", \\"unsupported_encoding\\"}\\n    decode_bad = any(count for name, count in decode_errors.items() if name not in decode_blocked_categories)\\n    result[\\"decoding\\"][\\"status\\"] = \\"FAIL\\" if decode_bad else \\"BLOCKED\\" if decode_errors or skips else \\"PASS\\" if result[\\"decoding\\"][\\"successes\\"] else \\"BLOCKED\\" if config.max_pixel_decodes > 0 else \\"NOT_ASSESSED\\"\\n    result[\\"status\\"] = \\"FAIL\\" if header_bad or decode_bad or identity_fail or result[\\"geometry\\"][\\"status\\"] == \\"FAIL\\" or malformed else \\"BLOCKED\\" if header_errors or result[\\"decoding\\"][\\"status\\"] == \\"BLOCKED\\" or not result[\\"headers\\"][\\"successes\\"] else \\"PASS\\"\\n    if result[\\"time_limit_reached\\"] and result[\\"status\\"] != \\"FAIL\\":\\n        result[\\"status\\"] = \\"BLOCKED\\"\\n    result[\\"runtime_seconds\\"] = time.perf_counter() - started\\n    return result\\n","inputs":"\\"\\"\\"Bounded read-only discovery and CSV reads with explicit incomplete states.\\"\\"\\"\\r\\n\\r\\nimport csv\\r\\nfrom datetime import datetime, timezone\\r\\nimport hashlib\\r\\nimport io\\r\\nimport os\\r\\nfrom pathlib import Path\\r\\nimport stat\\r\\n\\r\\nfrom .config import AuditConfig\\r\\n\\r\\n\\r\\nMANIFEST_NAMES = (\\"train.csv\\", \\"train_series.csv\\", \\"test.csv\\", \\"test_series.csv\\", \\"sample_submission.csv\\")\\r\\nIMAGE_DIRECTORIES = {\\"train_series\\", \\"test_series\\", \\"train_images\\", \\"test_images\\"}\\r\\nPROTECTED_COMPONENTS = {\\".git\\", \\".aws\\", \\".kaggle\\", \\".codex\\", \\".venv\\", \\"venv\\", \\"node_modules\\", \\"credentials\\", \\"access_token\\", \\"kaggle.json\\"}\\r\\n\\r\\n\\r\\nclass InputError(ValueError):\\r\\n    \\"\\"\\"Input contracts or safe-path requirements were not satisfied.\\"\\"\\"\\r\\n\\r\\n\\r\\ndef safe_absolute(path: str | Path) -> Path:\\r\\n    \\"\\"\\"Reject links/reparse ancestors rather than silently resolving through them.\\"\\"\\"\\r\\n    absolute = Path(os.path.abspath(path))\\r\\n    if any(part.casefold() in PROTECTED_COMPONENTS or part.casefold().startswith(\\".env\\") for part in absolute.parts):\\r\\n        raise InputError(\\"Protected directory is not a permitted input or output path\\")\\r\\n    for ancestor in (absolute, *absolute.parents):\\r\\n        try:\\r\\n            info = ancestor.lstat()\\r\\n        except FileNotFoundError:\\r\\n            continue\\r\\n        if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n            raise InputError(\\"Symbolic links and reparse points are not permitted\\")\\r\\n    return absolute\\r\\n\\r\\n\\r\\ndef discover_inputs(config: AuditConfig) -> dict:\\r\\n    result = {\\r\\n        \\"status\\": \\"INPUTS_NOT_LOCATED\\", \\"input_base\\": None, \\"selected_root\\": None,\\r\\n        \\"directories_checked\\": 0, \\"entries_checked\\": 0, \\"scan_complete\\": True,\\r\\n        \\"limit_reached\\": False, \\"errors\\": [], \\"error_count\\": 0,\\r\\n        \\"skipped_links\\": [], \\"skipped_link_count\\": 0, \\"candidates\\": [],\\r\\n        \\"csv_contents_read\\": False, \\"dicom_contents_read\\": False,\\r\\n        \\"dataset_identity_verified\\": False,\\r\\n    }\\r\\n\\r\\n    def issue(path, error):\\r\\n        result[\\"error_count\\"] += 1\\r\\n        if len(result[\\"errors\\"]) < 32:\\r\\n            result[\\"errors\\"].append({\\"path\\": str(path), \\"type\\": type(error).__name__, \\"errno\\": getattr(error, \\"errno\\", None)})\\r\\n\\r\\n    try:\\r\\n        base = safe_absolute(config.input_base)\\r\\n        result[\\"input_base\\"] = str(base)\\r\\n        start = safe_absolute(config.data_root) if config.data_root else base\\r\\n        if not start.is_relative_to(base):\\r\\n            raise InputError(\\"Explicit data_root must be within input_base\\")\\r\\n        info = start.lstat()\\r\\n        if not stat.S_ISDIR(info.st_mode):\\r\\n            raise InputError(\\"Input root is not a directory\\")\\r\\n    except FileNotFoundError:\\r\\n        result[\\"status\\"] = \\"INPUT_ROOT_MISSING\\" if config.data_root else \\"INPUT_BASE_MISSING\\"\\r\\n        return result\\r\\n    except (OSError, InputError) as error:\\r\\n        issue(config.input_base, error)\\r\\n        result.update(status=\\"INPUT_PATH_INVALID\\", scan_complete=False)\\r\\n        return result\\r\\n\\r\\n    def checked_stat(path):\\r\\n        try:\\r\\n            info = path.lstat()\\r\\n            if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\r\\n                result[\\"skipped_link_count\\"] += 1\\r\\n                if len(result[\\"skipped_links\\"]) < 32:\\r\\n                    result[\\"skipped_links\\"].append(str(path))\\r\\n                return None\\r\\n            if not path.resolve().is_relative_to(base):\\r\\n                raise InputError(\\"Discovered path escaped input_base\\")\\r\\n            return info\\r\\n        except FileNotFoundError:\\r\\n            return None\\r\\n        except (OSError, InputError) as error:\\r\\n            issue(path, error)\\r\\n            return None\\r\\n\\r\\n    pending = [(start, 0)]\\r\\n    cursor = 0\\r\\n    while cursor < len(pending) and result[\\"directories_checked\\"] < config.max_directories:\\r\\n        directory, depth = pending[cursor]\\r\\n        cursor += 1\\r\\n        result[\\"directories_checked\\"] += 1\\r\\n        files = []\\r\\n        for name in MANIFEST_NAMES:\\r\\n            path = directory / name\\r\\n            info = checked_stat(path)\\r\\n            if info is not None and stat.S_ISREG(info.st_mode):\\r\\n                files.append({\\"name\\": name, \\"path\\": str(path), \\"size_bytes\\": info.st_size,\\r\\n                              \\"modified_utc\\": datetime.fromtimestamp(info.st_mtime, timezone.utc).isoformat(), \\"sha256\\": None})\\r\\n        if files:\\r\\n            present = {item[\\"name\\"] for item in files}\\r\\n            images = {}\\r\\n            for name in sorted(IMAGE_DIRECTORIES):\\r\\n                path = directory / name\\r\\n                info = checked_stat(path)\\r\\n                if info is not None and stat.S_ISDIR(info.st_mode):\\r\\n                    images[name] = str(path)\\r\\n            result[\\"candidates\\"].append({\\"root\\": str(directory), \\"files\\": files,\\r\\n                                         \\"missing\\": [name for name in MANIFEST_NAMES if name not in present],\\r\\n                                         \\"image_directory_candidates\\": images, \\"dicom_file_count\\": None})\\r\\n        if config.data_root is not None or depth >= config.max_depth:\\r\\n            continue\\r\\n        try:\\r\\n            with os.scandir(directory) as entries:\\r\\n                for entry in entries:\\r\\n                    if result[\\"entries_checked\\"] >= config.max_entries:\\r\\n                        result[\\"limit_reached\\"] = True\\r\\n                        break\\r\\n                    result[\\"entries_checked\\"] += 1\\r\\n                    if entry.name.startswith(\\".\\") or entry.name in IMAGE_DIRECTORIES or entry.name.casefold() in PROTECTED_COMPONENTS:\\r\\n                        continue\\r\\n                    child = directory / entry.name\\r\\n                    info = checked_stat(child)\\r\\n                    if info is not None and stat.S_ISDIR(info.st_mode):\\r\\n                        pending.append((child, depth + 1))\\r\\n        except OSError as error:\\r\\n            issue(directory, error)\\r\\n        if result[\\"limit_reached\\"]:\\r\\n            break\\r\\n    result[\\"scan_complete\\"] = cursor == len(pending) and not result[\\"limit_reached\\"] and not result[\\"error_count\\"]\\r\\n    result[\\"candidates\\"].sort(key=lambda item: item[\\"root\\"])\\r\\n    complete = [item for item in result[\\"candidates\\"] if not item[\\"missing\\"]]\\r\\n    if len(complete) > 1:\\r\\n        result[\\"status\\"] = \\"AMBIGUOUS_INPUT_ROOTS\\"\\r\\n    elif not result[\\"scan_complete\\"]:\\r\\n        result[\\"status\\"] = \\"DISCOVERY_INCOMPLETE\\"\\r\\n    elif complete:\\r\\n        result.update(status=\\"MANIFESTS_LOCATED\\", selected_root=complete[0][\\"root\\"])\\r\\n    elif result[\\"candidates\\"]:\\r\\n        result[\\"status\\"] = \\"INCOMPLETE_MANIFESTS\\"\\r\\n    return result\\r\\n\\r\\n\\r\\ndef read_csv_manifest(path: str | Path, config: AuditConfig) -> tuple[list[str], list[dict], str]:\\r\\n    source = safe_absolute(path)\\r\\n    base = safe_absolute(config.input_base)\\r\\n    if not source.is_relative_to(base) or source.name not in MANIFEST_NAMES:\\r\\n        raise InputError(\\"CSV source must be a named competition manifest within input_base\\")\\r\\n    if not stat.S_ISREG(source.lstat().st_mode):\\r\\n        raise InputError(\\"CSV source must be a regular file\\")\\r\\n    with source.open(\\"rb\\") as stream:\\r\\n        if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):\\r\\n            raise InputError(\\"CSV source must be a regular file\\")\\r\\n        raw = stream.read(config.max_file_bytes + 1)\\r\\n    if len(raw) > config.max_file_bytes:\\r\\n        raise InputError(\\"CSV_FILE_LIMIT_EXCEEDED\\")\\r\\n    try:\\r\\n        reader = csv.DictReader(io.StringIO(raw.decode(\\"utf-8-sig\\"), newline=\\"\\"), strict=True)\\r\\n        columns = reader.fieldnames\\r\\n        if not columns or any(not name for name in columns) or len(set(columns)) != len(columns):\\r\\n            raise InputError(\\"CSV_HEADERS_MISSING_OR_DUPLICATED\\")\\r\\n        rows = []\\r\\n        for number, row in enumerate(reader, start=1):\\r\\n            if number > config.max_csv_rows:\\r\\n                raise InputError(\\"CSV_ROW_LIMIT_EXCEEDED\\")\\r\\n            if None in row or any(value is None for value in row.values()):\\r\\n                raise InputError(\\"CSV_ROW_WIDTH_INVALID\\")\\r\\n            rows.append(row)\\r\\n    except (UnicodeError, csv.Error) as error:\\r\\n        raise InputError(\\"CSV_ENCODING_OR_FORMAT_INVALID\\") from error\\r\\n    return columns, rows, hashlib.sha256(raw).hexdigest()\\r\\n","metrics":"\\"\\"\\"Observed-label ROC AUC with explicit support and undefined macro metrics.\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport math\\r\\nfrom collections.abc import Iterable, Mapping, Sequence\\r\\nfrom typing import Any\\r\\n\\r\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, parse_label, validate_identifier\\r\\nfrom .submission import validate_submission\\r\\n\\r\\n\\r\\ndef binary_auc(labels: Iterable[Any], scores: Iterable[Any]) -> float | None:\\r\\n    \\"\\"\\"Compute rank AUC, averaging tied ranks; return None for missing classes.\\"\\"\\"\\r\\n    observed_labels = list(labels)\\r\\n    observed_scores = list(scores)\\r\\n    if len(observed_labels) != len(observed_scores):\\r\\n        raise ContractError(\\"AUC labels and scores must have identical lengths\\")\\r\\n    pairs = []\\r\\n    for label, score in zip(observed_labels, observed_scores):\\r\\n        if not _numeric(label) or parse_label(label)[1] != \\"observed\\":\\r\\n            raise ContractError(\\"AUC requires observed numeric binary labels\\")\\r\\n        if not _numeric(score):\\r\\n            raise ContractError(\\"AUC requires numeric scores\\")\\r\\n        try:\\r\\n            numeric_score = float(score)\\r\\n        except (OverflowError, ValueError) as error:\\r\\n            raise ContractError(\\"AUC requires finite scores\\") from error\\r\\n        if not math.isfinite(numeric_score):\\r\\n            raise ContractError(\\"AUC requires finite scores\\")\\r\\n        pairs.append((numeric_score, float(label)))\\r\\n    positive_count = sum(label == 1.0 for _, label in pairs)\\r\\n    negative_count = len(pairs) - positive_count\\r\\n    if not positive_count or not negative_count:\\r\\n        return None\\r\\n    pairs.sort(key=lambda pair: pair[0])\\r\\n    positive_rank_sum = 0.0\\r\\n    start = 0\\r\\n    while start < len(pairs):\\r\\n        end = start + 1\\r\\n        while end < len(pairs) and pairs[end][0] == pairs[start][0]:\\r\\n            end += 1\\r\\n        mean_rank = (start + 1 + end) / 2.0\\r\\n        positive_rank_sum += mean_rank * sum(label == 1.0 for _, label in pairs[start:end])\\r\\n        start = end\\r\\n    return (positive_rank_sum - positive_count * (positive_count + 1) / 2.0) / (positive_count * negative_count)\\r\\n\\r\\n\\r\\ndef evaluate_predictions(\\r\\n    label_rows: Sequence[Mapping[str, Any]],\\r\\n    prediction_rows: Sequence[Mapping[str, Any]],\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Match studies exactly and evaluate each target on observed labels only.\\"\\"\\"\\r\\n    if not label_rows:\\r\\n        raise ContractError(\\"reference label table must not be empty\\")\\r\\n    references: dict[str, Mapping[str, Any]] = {}\\r\\n    for index, row in enumerate(label_rows):\\r\\n        if not isinstance(row, Mapping) or not set((STUDY_ID, *TARGETS)).issubset(row):\\r\\n            raise ContractError(f\\"reference row {index} is missing required cells\\")\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in references:\\r\\n            raise ContractError(f\\"duplicate reference identifier at row {index}\\")\\r\\n        references[uid] = row\\r\\n    validate_submission(prediction_rows, references)\\r\\n    predictions = {row[STUDY_ID]: row for row in prediction_rows}\\r\\n    per_target = {}\\r\\n    for target in TARGETS:\\r\\n        labels = []\\r\\n        scores = []\\r\\n        unknown_count = 0\\r\\n        for uid, row in references.items():\\r\\n            label, state = parse_label(row[target])\\r\\n            if state == \\"invalid\\":\\r\\n                raise ContractError(f\\"reference contains an invalid label for {target}\\")\\r\\n            if state == \\"unknown\\":\\r\\n                unknown_count += 1\\r\\n                continue\\r\\n            labels.append(label)\\r\\n            scores.append(predictions[uid][target])\\r\\n        positives = sum(label == 1.0 for label in labels)\\r\\n        per_target[target] = {\\r\\n            \\"observed\\": len(labels), \\"positive\\": positives,\\r\\n            \\"negative\\": len(labels) - positives, \\"unknown\\": unknown_count,\\r\\n            \\"auc\\": binary_auc(labels, scores),\\r\\n        }\\r\\n    defined = [stats[\\"auc\\"] for stats in per_target.values() if stats[\\"auc\\"] is not None]\\r\\n    complete = len(defined) == len(TARGETS)\\r\\n    return {\\r\\n        \\"status\\": \\"FULL_METRICS_AVAILABLE\\" if complete else \\"INCOMPLETE_METRICS\\",\\r\\n        \\"row_count\\": len(references), \\"per_target\\": per_target,\\r\\n        \\"macro_auc\\": sum(defined) / len(TARGETS) if complete else None,\\r\\n        \\"diagnostic_defined_target_macro_auc\\": sum(defined) / len(defined) if defined else None,\\r\\n        \\"defined_target_count\\": len(defined), \\"required_target_count\\": len(TARGETS),\\r\\n    }\\r\\n","phase_b":"\\"\\"\\"Read-only, bounded real-data audit; execution never grants training approval.\\"\\"\\"\\n\\nfrom datetime import datetime, timezone\\nimport json\\nfrom pathlib import Path\\nimport platform\\nimport time\\nimport uuid\\n\\nfrom .audit import run_preflight\\nfrom .contracts import ContractError\\nfrom .inputs import InputError, MANIFEST_NAMES, read_csv_manifest, safe_absolute\\nfrom .phase_b_config import PhaseBConfig\\n\\n\\nCHECK_STATES = {\\"PASS\\", \\"FAIL\\", \\"BLOCKED\\", \\"NOT_ASSESSED\\"}\\n\\n\\ndef _private_preflight_summary(preflight):\\n    \\"\\"\\"Retain measured contracts/hashes, not traversal paths or unbounded categories.\\"\\"\\"\\n    contracts = preflight[\\"csv_contracts\\"]\\n    if contracts.get(\\"status\\") == \\"VALID\\":\\n        contracts = json.loads(json.dumps(contracts, allow_nan=False))\\n        # Input metadata categories may contain arbitrary user strings. Preserve\\n        # observed/unknown/invalid counts here; sampling stores typed strata privately.\\n        for split in (\\"train_series\\", \\"test_series\\"):\\n            for stats in contracts[split][\\"metadata_statistics\\"].values():\\n                stats.pop(\\"value_counts\\", None)\\n    else:\\n        contracts = {key: contracts[key] for key in (\\"status\\", \\"error_type\\") if key in contracts}\\n    discovery = preflight[\\"discovery\\"]\\n    return {\\n        \\"status\\": preflight[\\"status\\"], \\"csv_contracts\\": contracts,\\n        \\"discovery\\": {key: discovery[key] for key in (\\n            \\"status\\", \\"directories_checked\\", \\"entries_checked\\", \\"scan_complete\\",\\n            \\"limit_reached\\", \\"error_count\\", \\"skipped_link_count\\")},\\n        \\"scope\\": \\"All five bounded CSV reads and contracts; no hidden/private test targets inspected\\",\\n    }\\n\\n\\ndef _state(summary, *, complete_key=None):\\n    state = summary.get(\\"status\\")\\n    if state in CHECK_STATES:\\n        return state\\n    if \\"INVALID\\" in str(state):\\n        return \\"FAIL\\"\\n    if complete_key and not summary.get(complete_key, False):\\n        return \\"BLOCKED\\"\\n    if \\"LIMITED\\" in str(state):\\n        return \\"BLOCKED\\"\\n    return \\"PASS\\"\\n\\n\\ndef _inventory_state(summary):\\n    statistics = summary.get(\\"split_statistics\\", {}).values()\\n    if (summary.get(\\"ancestry_mismatches\\", 0)\\n            or any(stats.get(\\"ambiguous_series\\", 0) or stats.get(\\"missing_series\\", 0) for stats in statistics)):\\n        return \\"FAIL\\"\\n    return _state(summary, complete_key=\\"scan_complete\\")\\n\\n\\ndef run_phase_b(config: PhaseBConfig, source_identity: dict | None = None, *, preflight: dict | None = None) -> dict:\\n    if not isinstance(config, PhaseBConfig):\\n        raise TypeError(\\"run_phase_b requires validated PhaseBConfig\\")\\n    started = time.perf_counter()\\n    csv_config = config.preflight_config()\\n    if preflight is None:\\n        preflight = run_preflight(csv_config, source_identity=source_identity)\\n    elif (preflight.get(\\"configuration\\") != csv_config.to_dict()\\n          or preflight.get(\\"source_identity\\") != (source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"})):\\n        raise InputError(\\"CSV preflight is stale; rerun preflight with the current configuration and source\\")\\n    checks = {name: {\\"status\\": \\"NOT_ASSESSED\\"} for name in (\\n        \\"series_inventory\\", \\"series_file_sample\\", \\"dicom\\", \\"dicom_headers\\", \\"pixel_decoding\\", \\"report_quality\\", \\"validation_support\\")}\\n    csv_valid = preflight[\\"csv_contracts\\"].get(\\"status\\") == \\"VALID\\"\\n    checks[\\"csv_preflight\\"] = {\\"status\\": \\"PASS\\" if csv_valid else (\\n        \\"FAIL\\" if preflight[\\"csv_contracts\\"].get(\\"status\\") == \\"INVALID\\" else \\"BLOCKED\\")}\\n    input_hash_recheck = {\\"status\\": \\"NOT_ASSESSED\\"}\\n    if csv_valid:\\n        # Imported only when preflight establishes an unambiguous permitted root.\\n        from .series_inventory import inventory_series, sample_series_files\\n        from .dicom_audit import inspect_dicom\\n        from .report_quality import aggregate_reports\\n        from .validation_support import summarize_validation\\n\\n        root = safe_absolute(preflight[\\"discovery\\"][\\"selected_root\\"])\\n        try:\\n            tables, current_hashes = {}, {}\\n            for name in MANIFEST_NAMES:\\n                _, tables[name], current_hashes[name] = read_csv_manifest(root / name, csv_config)\\n            if current_hashes != preflight[\\"csv_contracts\\"][\\"sha256\\"]:\\n                raise InputError(\\"CSV_CHANGED_SINCE_PREFLIGHT\\")\\n            inventory, records = inventory_series(root, tables[\\"train_series.csv\\"], tables[\\"test_series.csv\\"], config)\\n            sample, files = sample_series_files(records, config)\\n            checks[\\"series_inventory\\"] = {**inventory, \\"status\\": _inventory_state(inventory)}\\n            sample_state = _state(sample)\\n            if sample.get(\\"empty_series\\", 0):\\n                sample_state = \\"FAIL\\"\\n            elif (not files or sample.get(\\"series_with_unscanned_subdirectories\\", 0)\\n                  or sample.get(\\"series_with_no_candidate_dicom_filenames_in_immediate_entries\\", 0)):\\n                sample_state = \\"BLOCKED\\"\\n            checks[\\"series_file_sample\\"] = {**sample, \\"status\\": sample_state}\\n            dicom = inspect_dicom(files, config)\\n            checks[\\"dicom\\"] = dicom\\n            for name, key in ((\\"dicom_headers\\", \\"headers\\"), (\\"pixel_decoding\\", \\"decoding\\")):\\n                detail = dicom[key]\\n                measured_state = detail[\\"status\\"]\\n                if measured_state == \\"NOT_ASSESSED\\" and (name == \\"dicom_headers\\" or config.max_pixel_decodes):\\n                    measured_state = \\"BLOCKED\\"\\n                checks[name] = {\\"status\\": measured_state, \\"attempts\\": detail[\\"attempts\\"],\\n                                \\"successes\\": detail[\\"successes\\"], \\"failures\\": detail[\\"failures\\"],\\n                                \\"scope\\": \\"Inventory-selected sample only\\"}\\n            checks[\\"report_quality\\"] = aggregate_reports(tables[\\"train.csv\\"])\\n            checks[\\"validation_support\\"] = summarize_validation(tables[\\"train.csv\\"], preflight[\\"csv_contracts\\"][\\"train\\"])\\n            after_hashes = {name: read_csv_manifest(root / name, csv_config)[2] for name in MANIFEST_NAMES}\\n            input_hash_recheck = {\\"status\\": \\"PASS\\" if after_hashes == current_hashes else \\"FAIL\\",\\n                                  \\"scope\\": \\"All five CSVs SHA-256 before/after; DICOMs opened read-only, not globally hashed\\",\\n                                  \\"csv_sha256\\": after_hashes}\\n        except (InputError, ContractError, OSError) as error:\\n            checks[\\"input_read\\"] = {\\"status\\": \\"BLOCKED\\", \\"error_category\\": type(error).__name__,\\n                                    \\"scope\\": \\"No raw exception text, paths, or report text retained\\"}\\n    checks[\\"input_immutability\\"] = input_hash_recheck\\n    for summary in checks.values():\\n        if summary[\\"status\\"] not in CHECK_STATES:\\n            raise ValueError(\\"Phase B checks must declare a recognized status\\")\\n    states = {summary[\\"status\\"] for summary in checks.values()}\\n    status = \\"FAIL\\" if \\"FAIL\\" in states else \\"BLOCKED\\" if {\\"BLOCKED\\", \\"NOT_ASSESSED\\"} & states else \\"PASS\\"\\n    report = {\\n        \\"schema\\": \\"rsna-knee-phase-b-v1\\", \\"phase\\": \\"PHASE_B\\", \\"status\\": status,\\n        \\"created_utc\\": datetime.now(timezone.utc).isoformat(), \\"configuration\\": config.to_dict(),\\n        \\"source_identity\\": source_identity or {\\"status\\": \\"NOT_SUPPLIED\\"},\\n        \\"runtime\\": {\\"python\\": platform.python_version(), \\"platform\\": platform.system(),\\n                    \\"gpu_used\\": False, \\"elapsed_seconds\\": time.perf_counter() - started},\\n        \\"preflight\\": _private_preflight_summary(preflight), \\"checks\\": checks,\\n        \\"training_ready\\": False, \\"split_approved\\": False, \\"model_metrics\\": None,\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"independent_resource_verified\\": False},\\n        \\"scope\\": \\"Manifest consistency where exhaustive; bounded series/header/pixel samples otherwise. Visible test data are only the attached runtime sample.\\",\\n        \\"unassessed\\": [\\"full-dataset DICOM integrity\\", \\"complete codec support\\", \\"dataset-wide patient grouping\\",\\n                       \\"independent gold provenance\\", \\"approved twelve-target validation\\", \\"model performance\\"],\\n        \\"next_gate\\": \\"Review audit evidence and validation design before any training approval.\\",\\n    }\\n    # Enforce the output budget before the notebook prints any JSON as well as\\n    # at the write boundary. Never dump a large report then fail on its write.\\n    if len((json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + \\"\\\\n\\").encode(\\"utf-8\\")) > config.max_report_bytes:\\n        raise InputError(\\"REPORT_SIZE_LIMIT_EXCEEDED\\")\\n    return report\\n\\n\\ndef write_phase_b_report(report: dict, config: PhaseBConfig) -> Path:\\n    if not isinstance(config, PhaseBConfig):\\n        raise TypeError(\\"write_phase_b_report requires validated PhaseBConfig\\")\\n    output, source = safe_absolute(config.output_dir), safe_absolute(config.input_base)\\n    if output.is_relative_to(source) or source.is_relative_to(output):\\n        raise InputError(\\"Phase B outputs must be separate from input_base\\")\\n    payload = json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + \\"\\\\n\\"\\n    if len(payload.encode(\\"utf-8\\")) > config.max_report_bytes:\\n        raise InputError(\\"REPORT_SIZE_LIMIT_EXCEEDED\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    safe_absolute(output)\\n    stamp = datetime.now(timezone.utc).strftime(\\"%Y%m%dT%H%M%S%fZ\\")\\n    destination = output / f\\"phase_b_{stamp}_{uuid.uuid4().hex[:8]}.json\\"\\n    with destination.open(\\"x\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\") as stream:\\n        stream.write(payload)\\n    return destination\\n","phase_b_config":"\\"\\"\\"Validated, conservative CPU-only budgets for the read-only Phase B audit.\\"\\"\\"\\n\\nfrom dataclasses import dataclass, fields\\n\\nfrom .config import AuditConfig, ConfigError, _read_config_values\\n\\n\\n@dataclass(frozen=True)\\nclass PhaseBConfig(AuditConfig):\\n    inspect_csv: bool = True\\n    max_file_bytes: int = 32 * 1024 * 1024\\n    max_inventory_depth: int = 6\\n    max_inventory_directories: int = 60000\\n    max_inventory_entries: int = 1200000\\n    max_inventory_seconds: int = 180\\n    max_sample_series: int = 32\\n    max_files_per_series: int = 8\\n    max_sample_file_entries: int = 4096\\n    max_sample_total_entries: int = 32768\\n    max_header_reads: int = 128\\n    max_header_bytes: int = 2 * 1024 * 1024\\n    max_dicom_file_bytes: int = 64 * 1024 * 1024\\n    max_pixel_decodes: int = 12\\n    max_pixel_elements: int = 4 * 1024 * 1024\\n    max_dicom_seconds: int = 90\\n\\n    def __post_init__(self):\\n        super().__post_init__()\\n        if not self.inspect_csv:\\n            raise ConfigError(\\"Phase B requires CSV inspection\\")\\n        limits = {\\n            \\"max_inventory_depth\\": (1, 12),\\n            \\"max_inventory_directories\\": (1, 100000),\\n            \\"max_inventory_entries\\": (1, 2000000),\\n            \\"max_inventory_seconds\\": (1, 600),\\n            \\"max_sample_series\\": (1, 128),\\n            \\"max_files_per_series\\": (1, 32),\\n            \\"max_sample_file_entries\\": (1, 16384),\\n            \\"max_sample_total_entries\\": (1, 131072),\\n            \\"max_header_reads\\": (1, 512),\\n            \\"max_header_bytes\\": (1024, 8 * 1024 * 1024),\\n            \\"max_dicom_file_bytes\\": (1024, 128 * 1024 * 1024),\\n            \\"max_pixel_decodes\\": (0, 12),\\n            \\"max_pixel_elements\\": (1, 8 * 1024 * 1024),\\n            \\"max_dicom_seconds\\": (1, 300),\\n        }\\n        for name, (lower, upper) in limits.items():\\n            value = getattr(self, name)\\n            if type(value) is not int or not lower <= value <= upper:\\n                raise ConfigError(f\\"{name} must be an integer in [{lower}, {upper}]\\")\\n\\n    def preflight_config(self) -> AuditConfig:\\n        \\"\\"\\"Use the unchanged CSV/discovery contracts, without Phase B fields.\\"\\"\\"\\n        return AuditConfig.from_dict({field.name: getattr(self, field.name) for field in fields(AuditConfig)})\\n\\n\\ndef load_phase_b_config(path) -> PhaseBConfig:\\n    return PhaseBConfig.from_dict(_read_config_values(path))\\n","report_quality":"\\"\\"\\"Aggregate report-quality indicators without exporting report content.\\n\\nThe script categories below are character heuristics, not language detection.\\nInput rows and target labels are never changed or converted into weak labels.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom collections.abc import Mapping\\nimport math\\nimport statistics\\nimport unicodedata\\nfrom typing import Any\\n\\n\\nSHORT_REPORT_CHARACTERS = 20\\nLONG_REPORT_CHARACTERS = 10_000\\nSCRIPT_CATEGORIES = (\\n    \\"latin_only\\", \\"devanagari_only\\", \\"latin_and_devanagari\\",\\n    \\"latin_and_other_letters\\", \\"devanagari_and_other_letters\\",\\n    \\"latin_devanagari_and_other_letters\\", \\"other_letters_only\\", \\"no_letters\\",\\n)\\n\\n\\ndef _percentile(values: list[int], fraction: float) -> float | None:\\n    \\"\\"\\"Linearly interpolate sorted values at (n - 1) * fraction.\\"\\"\\"\\n    if not values:\\n        return None\\n    position = (len(values) - 1) * fraction\\n    lower, upper = math.floor(position), math.ceil(position)\\n    return values[lower] + (values[upper] - values[lower]) * (position - lower)\\n\\n\\ndef _distribution(values: list[int]) -> dict[str, int | float | None]:\\n    ordered = sorted(values)\\n    return {\\n        \\"count\\": len(ordered), \\"min\\": ordered[0] if ordered else None,\\n        \\"median\\": statistics.median(ordered) if ordered else None,\\n        \\"p90\\": _percentile(ordered, 0.90), \\"p99\\": _percentile(ordered, 0.99),\\n        \\"max\\": ordered[-1] if ordered else None,\\n    }\\n\\n\\ndef _script_category(latin: bool, devanagari: bool, other: bool) -> str:\\n    return {\\n        (True, False, False): \\"latin_only\\",\\n        (False, True, False): \\"devanagari_only\\",\\n        (True, True, False): \\"latin_and_devanagari\\",\\n        (True, False, True): \\"latin_and_other_letters\\",\\n        (False, True, True): \\"devanagari_and_other_letters\\",\\n        (True, True, True): \\"latin_devanagari_and_other_letters\\",\\n        (False, False, True): \\"other_letters_only\\",\\n        (False, False, False): \\"no_letters\\",\\n    }[(latin, devanagari, other)]\\n\\n\\ndef aggregate_reports(train_rows: list[dict[str, Any]]) -> dict[str, Any]:\\n    \\"\\"\\"Inspect all supplied Report cells; return fixed categories and counts.\\n\\n    The caller bounds manifest size before parsing. Duplicate normalization is\\n    NFKC, case folding and whitespace collapse, used in memory only. Empty\\n    reports do not contribute to duplicate or length distributions.\\n    \\"\\"\\"\\n    normalized: Counter[str] = Counter()\\n    scripts = {name: 0 for name in SCRIPT_CATEGORIES}\\n    flags = {name: 0 for name in (\\n        \\"unexpected_control_characters\\", \\"format_characters\\", \\"replacement_characters\\",\\n        \\"private_use_characters\\", \\"surrogate_characters\\", \\"unassigned_characters\\",\\n        \\"other_symbols\\", \\"unusual_characters\\", \\"very_short_reports\\", \\"very_long_reports\\",\\n    )}\\n    lengths: list[int] = []\\n    words: list[int] = []\\n    missing = empty = invalid = latin_reports = devanagari_reports = 0\\n    latin_letters = devanagari_letters = other_letters = 0\\n    latin_characters = devanagari_characters = 0\\n    for row in train_rows:\\n        if not isinstance(row, Mapping):\\n            invalid += 1\\n            continue\\n        report = row.get(\\"Report\\")\\n        if report is None:\\n            missing += 1\\n            continue\\n        if not isinstance(report, str):\\n            invalid += 1\\n            continue\\n        if not report.strip():\\n            empty += 1\\n            continue\\n        lengths.append(len(report))\\n        words.append(len(report.split()))\\n        normalized[\\" \\".join(unicodedata.normalize(\\"NFKC\\", report).casefold().split())] += 1\\n        encountered: set[str] = set()\\n        latin = devanagari = other = False\\n        for character in report:\\n            category = unicodedata.category(character)\\n            name = unicodedata.name(character, \\"\\")\\n            if \\"LATIN\\" in name:\\n                latin = True\\n                latin_characters += 1\\n            elif \\"DEVANAGARI\\" in name:\\n                devanagari = True\\n                devanagari_characters += 1\\n            if category.startswith(\\"L\\"):\\n                if \\"LATIN\\" in name:\\n                    latin_letters += 1\\n                elif \\"DEVANAGARI\\" in name:\\n                    devanagari_letters += 1\\n                else:\\n                    other = True\\n                    other_letters += 1\\n            if category == \\"Cc\\" and character not in \\"\\\\t\\\\r\\\\n\\":\\n                encountered.add(\\"unexpected_control_characters\\")\\n            if category == \\"Cf\\":\\n                encountered.add(\\"format_characters\\")\\n            if character == \\"\\\\ufffd\\":\\n                encountered.add(\\"replacement_characters\\")\\n            for expected, flag in ((\\"Co\\", \\"private_use_characters\\"), (\\"Cs\\", \\"surrogate_characters\\"),\\n                                   (\\"Cn\\", \\"unassigned_characters\\"), (\\"So\\", \\"other_symbols\\")):\\n                if category == expected:\\n                    encountered.add(flag)\\n        for flag in encountered:\\n            flags[flag] += 1\\n        flags[\\"unusual_characters\\"] += bool(encountered)\\n        flags[\\"very_short_reports\\"] += len(report.strip()) < SHORT_REPORT_CHARACTERS\\n        flags[\\"very_long_reports\\"] += len(report) > LONG_REPORT_CHARACTERS\\n        latin_reports += latin\\n        devanagari_reports += devanagari\\n        scripts[_script_category(latin, devanagari, other)] += 1\\n    duplicate_groups = sum(count > 1 for count in normalized.values())\\n    duplicates = sum(count - 1 for count in normalized.values())\\n    return {\\n        \\"status\\": \\"FAIL\\" if invalid else \\"PASS\\", \\"scope\\": \\"All supplied training Report cells\\",\\n        \\"row_count\\": len(train_rows), \\"nonempty_report_count\\": len(lengths),\\n        \\"missing_report_count\\": missing, \\"empty_report_count\\": empty,\\n        \\"empty_or_missing_report_count\\": missing + empty, \\"invalid_report_type_count\\": invalid,\\n        \\"normalized_duplicate_report_count\\": duplicates, \\"normalized_duplicate_group_count\\": duplicate_groups,\\n        \\"reports_in_duplicate_groups\\": sum(count for count in normalized.values() if count > 1),\\n        \\"normalization\\": \\"NFKC, casefold, whitespace collapse; nonempty reports only\\",\\n        \\"character_count\\": _distribution(lengths), \\"whitespace_word_count\\": _distribution(words),\\n        \\"percentile_method\\": \\"Linear interpolation at (n - 1) * quantile\\",\\n        \\"unicode_script_indicators\\": {\\n            \\"latin_present_report_count\\": latin_reports,\\n            \\"devanagari_present_report_count\\": devanagari_reports,\\n            \\"latin_character_count\\": latin_characters,\\n            \\"devanagari_character_count\\": devanagari_characters,\\n            \\"latin_letter_count\\": latin_letters, \\"devanagari_letter_count\\": devanagari_letters,\\n            \\"other_letter_count\\": other_letters, \\"heuristic_report_categories\\": scripts,\\n        },\\n        \\"quality_flag_report_counts\\": flags,\\n        \\"flag_thresholds\\": {\\"very_short_trimmed_characters_less_than\\": SHORT_REPORT_CHARACTERS,\\n                            \\"very_long_original_characters_greater_than\\": LONG_REPORT_CHARACTERS},\\n        \\"quality_review_required\\": bool(missing or empty or invalid or duplicates or any(flags.values())),\\n        \\"limitations\\": [\\n            \\"Unicode character names indicate scripts, not reliable language identification.\\",\\n            \\"Latin does not establish English; Devanagari does not establish Hindi.\\",\\n            \\"Whitespace tokens are not linguistic words and may undercount scripts without word spaces.\\",\\n            \\"Format characters and symbols can be legitimate; quality flags require human review.\\",\\n            \\"Normalized duplicates do not establish patient identity or report correctness.\\",\\n            \\"No report text, normalized strings, report-derived labels or identifiers are exported.\\",\\n        ],\\n        \\"weak_labels_created\\": False, \\"target_labels_changed\\": False,\\n    }\\n","series_inventory":"\\"\\"\\"Bounded series discovery and representative file sampling, without image reads.\\n\\nManifest identifiers are used only for private path matching. Public summaries\\ncontain counts and placeholder path conventions, never patient/study identifiers\\nor filesystem paths. Completeness concerns directory discovery or the selected\\nsample; this module never claims to count all DICOMs in the dataset.\\n\\"\\"\\"\\n\\nfrom collections import Counter, defaultdict, deque\\nimport os\\nfrom pathlib import Path\\nimport stat\\nimport time\\n\\nfrom .contracts import SERIES_ID, STUDY_ID, validate_identifier\\nfrom .inputs import InputError, PROTECTED_COMPONENTS, safe_absolute\\n\\n\\n_CONTAINERS = {\\"train_series\\": \\"train\\", \\"train_images\\": \\"train\\",\\n               \\"test_series\\": \\"test\\", \\"test_images\\": \\"test\\"}\\n_STRATA = (\\"Anatomical_Plane\\", \\"Fluid_Sensitive\\", \\"Fat_Suppression\\")\\n\\n\\ndef _validated_config(config):\\n    from .phase_b_config import PhaseBConfig\\n    if not isinstance(config, PhaseBConfig):\\n        raise TypeError(\\"series inventory requires validated PhaseBConfig\\")\\n\\n\\ndef _protected(name):\\n    return name.casefold() in PROTECTED_COMPONENTS or name.casefold().startswith(\\".env\\")\\n\\n\\ndef _checked(path, root):\\n    \\"\\"\\"Check lexical containment, links/reparse points, and ancestors each time.\\"\\"\\"\\n    absolute = Path(os.path.abspath(path))\\n    if not absolute.is_relative_to(root):\\n        raise InputError(\\"Discovered path escaped the selected root\\")\\n    info = absolute.lstat()\\n    if stat.S_ISLNK(info.st_mode) or getattr(info, \\"st_file_attributes\\", 0) & 0x400:\\n        return None\\n    safe_absolute(absolute)\\n    return info\\n\\n\\ndef _error(counter, error):\\n    # Error messages and filenames can contain private identifiers.\\n    counter[type(error).__name__] += 1\\n\\n\\ndef _bounded_entries(directory, budget, deadline):\\n    \\"\\"\\"Materialize at most budget entries; sort only this bounded collection.\\n\\n    At an exact cap the scan is conservatively incomplete: determining whether\\n    another entry exists would itself exceed the enumeration budget. A capped\\n    subset follows filesystem enumeration order and is not reproducible across\\n    filesystems, which callers must disclose.\\n    \\"\\"\\"\\n    entries = []\\n    complete = True\\n    reason = None\\n    with os.scandir(directory) as iterator:\\n        while True:\\n            if time.monotonic() >= deadline:\\n                complete, reason = False, \\"time\\"\\n                break\\n            if len(entries) >= budget:\\n                complete, reason = False, \\"entries\\"\\n                break\\n            try:\\n                entry = next(iterator)\\n            except StopIteration:\\n                break\\n            entries.append(entry.name)\\n    return sorted(entries), complete, reason\\n\\n\\ndef inventory_series(root, train_rows, test_rows, config):\\n    \\"\\"\\"Resolve manifest series by observed directory names with explicit limits.\\n\\n    Matching SeriesInstanceUID directories are pruned rather than enumerating\\n    their DICOM contents. If a known StudyInstanceUID or train/test container\\n    appears in the ancestry, it must agree with the manifest. Other layouts are\\n    permitted and learned from the paths actually observed.\\n    \\"\\"\\"\\n    _validated_config(config)\\n    started = time.monotonic()\\n    summary = {\\n        \\"status\\": \\"SERIES_INVENTORY_INVALID\\", \\"scan_complete\\": False,\\n        \\"directories_checked\\": 0, \\"entries_checked\\": 0,\\n        \\"matched_series_directories_pruned\\": 0, \\"skipped_links\\": 0,\\n        \\"skipped_protected\\": 0, \\"inaccessible_directories\\": 0,\\n        \\"ancestry_mismatches\\": 0, \\"limit_reasons\\": [],\\n        \\"reproducible_selection\\": True, \\"error_types\\": {},\\n        \\"split_statistics\\": {}, \\"root_layout\\": {}, \\"path_conventions\\": [],\\n        \\"dicom_file_count\\": None, \\"dicom_contents_read\\": False,\\n        \\"count_scope\\": \\"manifest series directory inventory; DICOM files are not enumerated\\",\\n        \\"completeness_scope\\": \\"directory namespace outside pruned matched-series payload subtrees\\",\\n        \\"uniqueness_scope\\": \\"observed series-directory matches in the traversed namespace\\",\\n        \\"elapsed_seconds\\": None,\\n        \\"budgets\\": {name: getattr(config, name) for name in (\\n            \\"max_inventory_depth\\", \\"max_inventory_directories\\", \\"max_inventory_entries\\", \\"max_inventory_seconds\\")},\\n    }\\n    try:\\n        base = safe_absolute(config.input_base)\\n        root = safe_absolute(root)\\n        if not root.is_relative_to(base):\\n            raise InputError(\\"Selected root must be within input_base\\")\\n        if not stat.S_ISDIR(root.lstat().st_mode):\\n            raise InputError(\\"Selected root must be a directory\\")\\n    except (OSError, InputError) as error:\\n        summary[\\"error_types\\"] = {type(error).__name__: 1}\\n        summary[\\"elapsed_seconds\\"] = round(max(0, time.monotonic() - started), 6)\\n        return summary, []\\n\\n    expected = {}\\n    studies = set()\\n    for split, rows in ((\\"train\\", train_rows), (\\"test\\", test_rows)):\\n        for row in rows:\\n            study = validate_identifier(row[STUDY_ID])\\n            series = validate_identifier(row[SERIES_ID])\\n            if series in expected:\\n                raise InputError(\\"Series identifiers must be unique across manifests\\")\\n            expected[series] = (split, row)\\n            studies.add(study)\\n\\n    hits = defaultdict(list)\\n    inaccessible = set()\\n    error_types = Counter()\\n    layout = Counter()\\n    conventions = Counter()\\n    limits = set()\\n    deadline = started + config.max_inventory_seconds\\n    pending = deque([(root, 0)])\\n\\n    def convention(path):\\n        parts = []\\n        for part in path.relative_to(root).parts:\\n            if part in expected:\\n                parts.append(\\"<SeriesInstanceUID>\\")\\n            elif part in studies:\\n                parts.append(\\"<StudyInstanceUID>\\")\\n            elif part.casefold() in _CONTAINERS:\\n                parts.append(part.casefold())\\n            else:\\n                parts.append(\\"<directory>\\")\\n        return \\"/\\".join(parts)\\n\\n    while pending:\\n        if time.monotonic() >= deadline:\\n            limits.add(\\"time\\")\\n            break\\n        if summary[\\"directories_checked\\"] >= config.max_inventory_directories:\\n            limits.add(\\"directories\\")\\n            break\\n        remaining = config.max_inventory_entries - summary[\\"entries_checked\\"]\\n        if remaining <= 0:\\n            limits.add(\\"entries\\")\\n            break\\n        directory, depth = pending.popleft()\\n        summary[\\"directories_checked\\"] += 1\\n        try:\\n            names, complete, reason = _bounded_entries(directory, remaining, deadline)\\n        except OSError as error:\\n            summary[\\"inaccessible_directories\\"] += 1\\n            _error(error_types, error)\\n            continue\\n        summary[\\"entries_checked\\"] += len(names)\\n        if not complete:\\n            limits.add(reason)\\n            summary[\\"reproducible_selection\\"] = False\\n        for name in names:\\n            if time.monotonic() >= deadline:\\n                limits.add(\\"time\\")\\n                break\\n            child = directory / name\\n            if _protected(name):\\n                summary[\\"skipped_protected\\"] += 1\\n                continue\\n            try:\\n                info = _checked(child, root)\\n            except (OSError, InputError) as error:\\n                _error(error_types, error)\\n                if name in expected:\\n                    inaccessible.add(name)\\n                continue\\n            if info is None:\\n                summary[\\"skipped_links\\"] += 1\\n                continue\\n            is_directory = stat.S_ISDIR(info.st_mode)\\n            if directory == root:\\n                kind = (name.casefold() if name.casefold() in _CONTAINERS else\\n                        \\"study_identifier_directory\\" if name in studies else\\n                        \\"series_identifier_directory\\" if name in expected else\\n                        \\"other_directory\\" if is_directory else \\"regular_file\\" if stat.S_ISREG(info.st_mode) else \\"other_entry\\")\\n                layout[kind] += 1\\n            if not is_directory:\\n                continue\\n            if depth >= config.max_inventory_depth:\\n                limits.add(\\"depth\\")\\n                continue\\n            if name in expected:\\n                summary[\\"matched_series_directories_pruned\\"] += 1\\n                split, row = expected[name]\\n                ancestor_parts = child.relative_to(root).parts[:-1]\\n                known_studies = {part for part in ancestor_parts if part in studies}\\n                known_splits = {_CONTAINERS[part.casefold()] for part in ancestor_parts if part.casefold() in _CONTAINERS}\\n                if (known_studies and known_studies != {row[STUDY_ID]}) or (known_splits and known_splits != {split}):\\n                    summary[\\"ancestry_mismatches\\"] += 1\\n                    continue\\n                # Opening a directory checks accessibility without listing its files.\\n                if summary[\\"directories_checked\\"] >= config.max_inventory_directories:\\n                    limits.add(\\"directories\\")\\n                    continue\\n                summary[\\"directories_checked\\"] += 1\\n                try:\\n                    with os.scandir(child):\\n                        pass\\n                except OSError as error:\\n                    inaccessible.add(name)\\n                    summary[\\"inaccessible_directories\\"] += 1\\n                    _error(error_types, error)\\n                    continue\\n                hits[name].append(child)\\n                conventions[(split, convention(child))] += 1\\n                continue\\n            if summary[\\"directories_checked\\"] + len(pending) >= config.max_inventory_directories:\\n                limits.add(\\"directories\\")\\n            else:\\n                pending.append((child, depth + 1))\\n        if not complete:\\n            break\\n\\n    # Skipped subtrees might conceal additional candidates, so even a unique\\n    # observed path cannot claim exhaustive uniqueness in an incomplete scan.\\n    summary[\\"scan_complete\\"] = not (limits or error_types or summary[\\"skipped_links\\"] or summary[\\"skipped_protected\\"])\\n    summary[\\"status\\"] = \\"SERIES_INVENTORY_COMPLETE\\" if summary[\\"scan_complete\\"] else \\"SERIES_INVENTORY_LIMITED\\"\\n    summary[\\"limit_reasons\\"] = sorted(limits)\\n    summary[\\"error_types\\"] = dict(sorted(error_types.items()))\\n    summary[\\"root_layout\\"] = dict(sorted(layout.items()))\\n    summary[\\"path_conventions\\"] = [\\n        {\\"split\\": split, \\"pattern\\": pattern, \\"observed_series_directories\\": count}\\n        for (split, pattern), count in sorted(conventions.items())\\n    ][:32]\\n    summary[\\"path_conventions_omitted\\"] = max(0, len(conventions) - 32)\\n    records = []\\n    for split in (\\"train\\", \\"test\\"):\\n        identifiers = [uid for uid, (row_split, _) in expected.items() if row_split == split]\\n        unlocated = sum(not hits[uid] and uid not in inaccessible for uid in identifiers)\\n        summary[\\"split_statistics\\"][split] = {\\n            \\"expected_series\\": len(identifiers),\\n            \\"located_series\\": sum(bool(hits[uid]) for uid in identifiers),\\n            \\"resolved_series\\": sum(len(hits[uid]) == 1 for uid in identifiers),\\n            \\"ambiguous_series\\": sum(len(hits[uid]) > 1 for uid in identifiers),\\n            \\"inaccessible_series\\": sum(uid in inaccessible for uid in identifiers),\\n            \\"missing_series\\": unlocated if summary[\\"scan_complete\\"] else None,\\n            \\"unlocated_not_proven_missing\\": unlocated if not summary[\\"scan_complete\\"] else 0,\\n        }\\n    for uid in sorted(expected):\\n        if len(hits[uid]) == 1:\\n            split, row = expected[uid]\\n            records.append({\\"split\\": split, \\"row\\": row, \\"path\\": hits[uid][0]})\\n    summary[\\"resolved_uniqueness_verified\\"] = summary[\\"scan_complete\\"]\\n    summary[\\"elapsed_seconds\\"] = round(max(0, time.monotonic() - started), 6)\\n    return summary, records\\n\\n\\ndef _stratum(record):\\n    return (record[\\"split\\"], *((type(record[\\"row\\"].get(field)).__name__, str(record[\\"row\\"].get(field))) for field in _STRATA))\\n\\n\\ndef sample_series_files(resolved_records, config):\\n    \\"\\"\\"Select stratified deterministic series and bounded DICOM file samples.\\"\\"\\"\\n    _validated_config(config)\\n    started = time.monotonic()\\n    summary = {\\n        \\"status\\": \\"SAMPLING_COMPLETE\\", \\"scan_complete\\": True,\\n        \\"reproducible_selection\\": True, \\"available_series\\": len(resolved_records),\\n        \\"selected_series\\": 0, \\"strata_count\\": 0, \\"strata_selected\\": 0,\\n        \\"stratification_fields\\": [\\"split\\", *_STRATA], \\"entries_checked\\": 0,\\n        \\"selected_files\\": 0, \\"candidate_dicom_files\\": 0, \\"unexpected_regular_files\\": 0,\\n        \\"unexpected_subdirectories\\": 0, \\"other_entries\\": 0, \\"skipped_links\\": 0,\\n        \\"skipped_protected\\": 0, \\"inaccessible_series\\": 0, \\"empty_series\\": 0,\\n        \\"limit_reasons\\": [], \\"error_types\\": {}, \\"representative_file_counts\\": [],\\n        \\"dicom_file_count\\": None, \\"dicom_contents_read\\": False,\\n        \\"count_scope\\": \\"immediate entries of selected series filename candidates only (.dcm/.dicom); extensions do not verify DICOM contents; incomplete per-series counts are lower bounds\\",\\n        \\"elapsed_seconds\\": None,\\n        \\"budgets\\": {name: getattr(config, name) for name in (\\n            \\"max_sample_series\\", \\"max_files_per_series\\", \\"max_sample_file_entries\\", \\"max_sample_total_entries\\", \\"max_header_reads\\", \\"max_dicom_seconds\\")},\\n        \\"split_selected_series\\": {\\"train\\": 0, \\"test\\": 0},\\n        \\"split_selected_files\\": {\\"train\\": 0, \\"test\\": 0},\\n        \\"extension_counts\\": {\\"dcm\\": 0, \\"dicom\\": 0, \\"other\\": 0},\\n        \\"series_without_candidate_dicom_files\\": 0,\\n        \\"series_with_no_candidate_dicom_filenames_in_immediate_entries\\": 0,\\n        \\"series_with_unscanned_subdirectories\\": 0,\\n        \\"unexpected_subdirectories_not_scanned\\": True,\\n    }\\n    base = safe_absolute(config.input_base)\\n    groups = defaultdict(list)\\n    for record in resolved_records:\\n        path = safe_absolute(record[\\"path\\"])\\n        if not path.is_relative_to(base):\\n            raise InputError(\\"Sample series path must be within input_base\\")\\n        groups[_stratum(record)].append(record)\\n    for records in groups.values():\\n        records.sort(key=lambda item: (item[\\"row\\"][STUDY_ID], item[\\"row\\"][SERIES_ID], str(item[\\"path\\"])))\\n    strata = sorted(groups)\\n    summary[\\"strata_count\\"] = len(strata)\\n    selected = []\\n    cursor = 0\\n    while len(selected) < config.max_sample_series:\\n        progressed = False\\n        for key in strata:\\n            if cursor < len(groups[key]):\\n                selected.append(groups[key][cursor])\\n                progressed = True\\n                if len(selected) == config.max_sample_series:\\n                    break\\n        if not progressed:\\n            break\\n        cursor += 1\\n    summary[\\"selected_series\\"] = len(selected)\\n    summary[\\"split_selected_series\\"] = {split: sum(record[\\"split\\"] == split for record in selected) for split in (\\"train\\", \\"test\\")}\\n    summary[\\"strata_selected\\"] = len({_stratum(record) for record in selected})\\n    summary[\\"series_selection_capped\\"] = len(selected) < len(resolved_records)\\n    error_types = Counter()\\n    limits = set()\\n    deadline = started + config.max_dicom_seconds\\n    candidate_groups = []\\n    for index, record in enumerate(selected):\\n        observed = {\\"sample_index\\": index, \\"split\\": record[\\"split\\"], \\"entries_checked\\": 0,\\n                    \\"candidate_dicom_files\\": 0, \\"selected_files\\": 0, \\"scan_complete\\": True,\\n                    \\"file_count_exact\\": True, \\"inaccessible\\": False,\\n                    \\"unexpected_subdirectories\\": 0}\\n        summary[\\"representative_file_counts\\"].append(observed)\\n        candidates = []\\n        candidate_groups.append(candidates)\\n        remaining = config.max_sample_total_entries - summary[\\"entries_checked\\"]\\n        budget = min(config.max_sample_file_entries, remaining)\\n        if budget <= 0 or time.monotonic() >= deadline:\\n            limits.add(\\"total_entries\\" if budget <= 0 else \\"time\\")\\n            observed.update(scan_complete=False, file_count_exact=False)\\n            continue\\n        path = Path(record[\\"path\\"])\\n        try:\\n            info = _checked(path, base)\\n            if info is None or not stat.S_ISDIR(info.st_mode):\\n                raise InputError(\\"Sample series is no longer a safe directory\\")\\n            names, complete, reason = _bounded_entries(path, budget, deadline)\\n        except (OSError, InputError) as error:\\n            summary[\\"inaccessible_series\\"] += 1\\n            _error(error_types, error)\\n            observed.update(scan_complete=False, file_count_exact=False, inaccessible=True)\\n            continue\\n        observed[\\"entries_checked\\"] = len(names)\\n        summary[\\"entries_checked\\"] += len(names)\\n        observed.update(scan_complete=complete, file_count_exact=complete)\\n        if not complete:\\n            limits.add(\\"per_series_entries\\" if reason == \\"entries\\" and budget == config.max_sample_file_entries else \\"total_entries\\" if reason == \\"entries\\" else reason)\\n            summary[\\"reproducible_selection\\"] = False\\n        for name in names:\\n            if time.monotonic() >= deadline:\\n                limits.add(\\"time\\")\\n                observed.update(scan_complete=False, file_count_exact=False)\\n                break\\n            if _protected(name):\\n                summary[\\"skipped_protected\\"] += 1\\n                observed.update(scan_complete=False, file_count_exact=False)\\n                continue\\n            child = path / name\\n            try:\\n                info = _checked(child, base)\\n            except (OSError, InputError) as error:\\n                _error(error_types, error)\\n                observed.update(scan_complete=False, file_count_exact=False)\\n                continue\\n            if info is None:\\n                summary[\\"skipped_links\\"] += 1\\n                observed.update(scan_complete=False, file_count_exact=False)\\n            elif stat.S_ISREG(info.st_mode):\\n                extension = child.suffix.casefold().lstrip(\\".\\")\\n                extension_type = extension if extension in {\\"dcm\\", \\"dicom\\"} else \\"other\\"\\n                summary[\\"extension_counts\\"][extension_type] += 1\\n                if extension_type in {\\"dcm\\", \\"dicom\\"}:\\n                    observed[\\"candidate_dicom_files\\"] += 1\\n                    summary[\\"candidate_dicom_files\\"] += 1\\n                    if len(candidates) < config.max_files_per_series:\\n                        candidates.append({\\"split\\": record[\\"split\\"], \\"row\\": record[\\"row\\"],\\n                                           \\"path\\": child, \\"series_path\\": path})\\n                else:\\n                    summary[\\"unexpected_regular_files\\"] += 1\\n            elif stat.S_ISDIR(info.st_mode):\\n                summary[\\"unexpected_subdirectories\\"] += 1\\n                observed[\\"unexpected_subdirectories\\"] += 1\\n            else:\\n                summary[\\"other_entries\\"] += 1\\n        if complete and not names:\\n            summary[\\"empty_series\\"] += 1\\n    # Round-robin file allocation keeps all selected strata represented before\\n    # assigning second files, even if the independent header budget is small.\\n    files = []\\n    for position in range(config.max_files_per_series):\\n        for index, candidates in enumerate(candidate_groups):\\n            if position < len(candidates) and len(files) < config.max_header_reads:\\n                files.append(candidates[position])\\n                summary[\\"representative_file_counts\\"][index][\\"selected_files\\"] += 1\\n    summary[\\"header_selection_capped\\"] = sum(len(group) for group in candidate_groups) > len(files)\\n    summary[\\"selected_files\\"] = len(files)\\n    summary[\\"split_selected_files\\"] = {split: sum(record[\\"split\\"] == split for record in files) for split in (\\"train\\", \\"test\\")}\\n    summary[\\"series_without_candidate_dicom_files\\"] = sum(not item[\\"candidate_dicom_files\\"] for item in summary[\\"representative_file_counts\\"])\\n    summary[\\"series_with_no_candidate_dicom_filenames_in_immediate_entries\\"] = sum(not item[\\"candidate_dicom_files\\"] and item[\\"file_count_exact\\"] and not item[\\"unexpected_subdirectories\\"] for item in summary[\\"representative_file_counts\\"])\\n    summary[\\"series_with_unscanned_subdirectories\\"] = sum(bool(item[\\"unexpected_subdirectories\\"]) for item in summary[\\"representative_file_counts\\"])\\n    summary[\\"scan_complete\\"] = not (limits or error_types or summary[\\"skipped_links\\"] or summary[\\"skipped_protected\\"])\\n    if not summary[\\"scan_complete\\"]:\\n        summary[\\"status\\"] = \\"SAMPLING_LIMITED\\"\\n    summary[\\"limit_reasons\\"] = sorted(limits)\\n    summary[\\"error_types\\"] = dict(sorted(error_types.items()))\\n    summary[\\"elapsed_seconds\\"] = round(max(0, time.monotonic() - started), 6)\\n    return summary, files\\n","submission":"\\"\\"\\"Validate the exact submission contract and write without overwriting files.\\"\\"\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport csv\\r\\nimport math\\r\\nfrom collections.abc import Iterable, Mapping, Sequence\\r\\nfrom pathlib import Path\\r\\nfrom typing import Any\\r\\n\\r\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, _numeric, validate_identifier\\r\\nfrom .inputs import safe_absolute\\r\\n\\r\\nSUBMISSION_COLUMNS = (STUDY_ID, *TARGETS)\\r\\n\\r\\n\\r\\ndef validate_submission(\\r\\n    rows: Sequence[Mapping[str, Any]],\\r\\n    test_ids: Iterable[str],\\r\\n    columns: Sequence[str] | None = None,\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Require each test study once and twelve finite probabilities in order.\\"\\"\\"\\r\\n    expected_ids = [validate_identifier(uid) for uid in test_ids]\\r\\n    if not expected_ids:\\r\\n        raise ContractError(\\"test identifier set must not be empty\\")\\r\\n    if len(set(expected_ids)) != len(expected_ids):\\r\\n        raise ContractError(\\"test identifier list contains duplicates\\")\\r\\n    if not rows:\\r\\n        raise ContractError(\\"submission must contain at least one row\\")\\r\\n    if not isinstance(rows[0], Mapping):\\r\\n        raise ContractError(\\"submission row 0 must be a mapping\\")\\r\\n    actual_columns = tuple(columns) if columns is not None else tuple(rows[0])\\r\\n    if actual_columns != SUBMISSION_COLUMNS:\\r\\n        raise ContractError(\\"submission columns must exactly match the identifier and target order\\")\\r\\n    observed_ids: set[str] = set()\\r\\n    for index, row in enumerate(rows):\\r\\n        if not isinstance(row, Mapping) or set(row) != set(SUBMISSION_COLUMNS):\\r\\n            raise ContractError(f\\"submission row {index} has missing or extra cells\\")\\r\\n        uid = validate_identifier(row[STUDY_ID])\\r\\n        if uid in observed_ids:\\r\\n            raise ContractError(f\\"duplicate submission identifier at row {index}\\")\\r\\n        observed_ids.add(uid)\\r\\n        for target in TARGETS:\\r\\n            value = row[target]\\r\\n            if not _numeric(value):\\r\\n                raise ContractError(f\\"submission row {index} has a nonnumeric probability for {target}\\")\\r\\n            try:\\r\\n                probability = float(value)\\r\\n            except (ValueError, OverflowError) as error:\\r\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\") from error\\r\\n            if not math.isfinite(probability) or not 0.0 <= probability <= 1.0:\\r\\n                raise ContractError(f\\"submission row {index} has an invalid probability for {target}\\")\\r\\n    if observed_ids != set(expected_ids):\\r\\n        raise ContractError(\\"submission identifier set differs from the exact test identifier set\\")\\r\\n    return {\\r\\n        \\"status\\": \\"PASS\\", \\"row_count\\": len(rows), \\"unique_study_count\\": len(observed_ids),\\r\\n        \\"columns\\": list(SUBMISSION_COLUMNS), \\"probabilities_per_row\\": len(TARGETS),\\r\\n    }\\r\\n\\r\\n\\r\\ndef write_submission(\\r\\n    rows: Sequence[Mapping[str, Any]],\\r\\n    test_ids: Iterable[str],\\r\\n    path: str | Path,\\r\\n    columns: Sequence[str] = SUBMISSION_COLUMNS,\\r\\n) -> dict[str, Any]:\\r\\n    \\"\\"\\"Validate, reject protected/link paths, and refuse any existing output.\\"\\"\\"\\r\\n    validation = validate_submission(rows, test_ids, columns)\\r\\n    destination = safe_absolute(path)\\r\\n    with destination.open(\\"x\\", newline=\\"\\", encoding=\\"utf-8\\") as stream:\\r\\n        writer = csv.DictWriter(stream, fieldnames=list(columns))\\r\\n        writer.writeheader()\\r\\n        writer.writerows(rows)\\r\\n    return {**validation, \\"path\\": str(destination), \\"size_bytes\\": destination.stat().st_size}\\r\\n","validation_support":"\\"\\"\\"Sparse-label support diagnostics, never an approved validation split.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Mapping\\nfrom typing import Any\\n\\nfrom .contracts import ContractError, STUDY_ID, TARGETS, label_vector, validate_identifier\\n\\n\\nEXACT_SEARCH_MAX_LABELED_STUDIES = {2: 12, 3: 8}\\n\\n\\ndef _joint_coverage(masks: list[int], folds: int) -> dict[str, Any]:\\n    \\"\\"\\"Find only a class-coverage witness on tiny study-grouped problems.\\n\\n    All twenty-four class bits must occur in every fold. No identity, row index\\n    or assignment is returned. Larger problems remain explicitly unassessed.\\n    The first study is fixed to fold zero to remove equivalent fold names.\\n    \\"\\"\\"\\n    limit = EXACT_SEARCH_MAX_LABELED_STUDIES[folds]\\n    base = {\\"scope\\": \\"Observed positive and negative class coverage only; study grouping\\",\\n            \\"labeled_study_count\\": len(masks), \\"exact_search_max_labeled_studies\\": limit,\\n            \\"split_approved\\": False}\\n    if len(masks) > limit:\\n        return {**base, \\"status\\": \\"NOT_ASSESSED\\", \\"reason\\": \\"Tiny exact-search limit exceeded; joint feasibility is not implied by marginal counts\\"}\\n    ordered = sorted(masks, key=lambda value: (-value.bit_count(), value))\\n    complete = (1 << (2 * len(TARGETS))) - 1\\n    suffix_union = [0] * (len(ordered) + 1)\\n    for index in range(len(ordered) - 1, -1, -1):\\n        suffix_union[index] = suffix_union[index + 1] | ordered[index]\\n    visits = 0\\n    failed: set[tuple[int, tuple[int, ...]]] = set()\\n\\n    def search(index: int, covered: tuple[int, ...], counts: tuple[int, ...]) -> tuple[int, ...] | None:\\n        nonlocal visits\\n        visits += 1\\n        if all(value == complete for value in covered):\\n            # Remaining labeled studies may be put in any fold without losing\\n            # coverage. This is not a balanced or patient-safe split proposal.\\n            return (counts[0] + len(ordered) - index, *counts[1:])\\n        if index == len(ordered) or any((value | suffix_union[index]) != complete for value in covered):\\n            return None\\n        key = (index, tuple(sorted(covered)))\\n        if key in failed:\\n            return None\\n        equivalent: set[int] = set()\\n        for fold, value in enumerate(covered):\\n            if value in equivalent:\\n                continue\\n            equivalent.add(value)\\n            updated = list(covered)\\n            updated[fold] |= ordered[index]\\n            sizes = list(counts)\\n            sizes[fold] += 1\\n            result = search(index + 1, tuple(updated), tuple(sizes))\\n            if result is not None:\\n                return result\\n        failed.add(key)\\n        return None\\n\\n    witness = search(1, (ordered[0], *([0] * (folds - 1))), (1, *([0] * (folds - 1)))) if ordered else None\\n    if witness is None:\\n        return {**base, \\"status\\": \\"FAIL\\", \\"reason\\": \\"Joint class coverage is infeasible in the exact study-grouped search\\",\\n                \\"search_states_visited\\": visits}\\n    return {**base, \\"status\\": \\"PASS\\", \\"reason\\": \\"Class-coverage witness found; this is not an approved validation design\\",\\n            \\"search_states_visited\\": visits, \\"witness_labeled_study_counts_per_fold\\": list(witness)}\\n\\n\\ndef _failure(reason: str) -> dict[str, Any]:\\n    return {\\"status\\": \\"FAIL\\", \\"reason\\": reason, \\"training_ready\\": False,\\n            \\"validation_design_status\\": \\"REVIEW_REQUIRED\\", \\"split_approved\\": False}\\n\\n\\ndef summarize_validation(train_rows: list[dict[str, Any]], csv_train_summary: dict[str, Any]) -> dict[str, Any]:\\n    \\"\\"\\"Reuse verified target statistics, preserving every unknown-label mask.\\n\\n    Rechecking counts against unchanged input rows protects this interface from\\n    a stale summary. The optional tiny exact search proves only class coverage,\\n    not balance, reliable estimates, patient safety or a suitable split.\\n    \\"\\"\\"\\n    if not isinstance(csv_train_summary, Mapping) or csv_train_summary.get(\\"status\\") != \\"PASS\\":\\n        return _failure(\\"Valid training CSV contract summary is required\\")\\n    if csv_train_summary.get(\\"row_count\\") != len(train_rows):\\n        return _failure(\\"CSV summary row count does not match supplied training rows\\")\\n    source = csv_train_summary.get(\\"target_statistics\\")\\n    if not isinstance(source, Mapping) or any(target not in source for target in TARGETS):\\n        return _failure(\\"CSV summary must include all twelve verified target statistics\\")\\n    observed = {target: {\\"positive\\": 0, \\"negative\\": 0, \\"observed\\": 0, \\"unknown\\": 0, \\"invalid\\": 0}\\n                for target in TARGETS}\\n    seen: set[str] = set()\\n    masks: list[int] = []\\n    try:\\n        for row in train_rows:\\n            uid = validate_identifier(row[STUDY_ID])\\n            if uid in seen:\\n                return _failure(\\"Study grouping requires unique training study identifiers\\")\\n            seen.add(uid)\\n            vector = label_vector(row)\\n            mask = 0\\n            for index, target in enumerate(TARGETS):\\n                state, value = vector[\\"states\\"][index], vector[\\"values\\"][index]\\n                observed[target][state] += 1\\n                if vector[\\"observed_mask\\"][index]:\\n                    name = \\"positive\\" if value == 1.0 else \\"negative\\"\\n                    observed[target][name] += 1\\n                    mask |= 1 << (2 * index + int(value == 1.0))\\n            if mask:\\n                masks.append(mask)\\n    except (ContractError, KeyError, TypeError):\\n        return _failure(\\"Supplied training rows violate study or label contracts\\")\\n    targets = {}\\n    for target in TARGETS:\\n        stats = source[target]\\n        if not isinstance(stats, Mapping):\\n            return _failure(\\"CSV target statistics are malformed\\")\\n        for key, actual in observed[target].items():\\n            value = stats.get(key)\\n            if type(value) is not int or value < 0 or value != actual:\\n                return _failure(\\"CSV target statistics do not match supplied label observations\\")\\n        if stats[\\"invalid\\"]:\\n            return _failure(\\"Invalid label values cannot establish validation support\\")\\n        # Reuse the verified counts; prevalence uses observed labels only.\\n        targets[target] = {key: stats[key] for key in (\\"positive\\", \\"negative\\", \\"observed\\", \\"unknown\\", \\"invalid\\")}\\n        targets[target][\\"observation_coverage\\"] = stats[\\"observed\\"] / len(train_rows) if train_rows else None\\n        targets[target][\\"support_consequence\\"] = (\\n            \\"Both classes observed; marginal support does not prove joint split feasibility or reliable estimates\\"\\n            if stats[\\"positive\\"] and stats[\\"negative\\"] else\\n            \\"At least one class has no observed support; a two-class validation estimate is unsupported\\"\\n        )\\n    diagnostics = {}\\n    for folds, name in ((2, \\"two_folds\\"), (3, \\"three_folds\\")):\\n        deficient = [target for target in TARGETS if min(targets[target][\\"positive\\"], targets[target][\\"negative\\"]) < folds]\\n        joint = {\\"status\\": \\"FAIL\\", \\"reason\\": \\"Necessary per-class study support is insufficient\\", \\"split_approved\\": False}\\n        if not deficient:\\n            joint = _joint_coverage(masks, folds)\\n        diagnostics[name] = {\\n            \\"fold_count\\": folds, \\"necessary_per_class_support\\": folds,\\n            \\"necessary_support_status\\": \\"FAIL\\" if deficient else \\"PASS\\",\\n            \\"targets_below_necessary_support\\": deficient, \\"joint_class_coverage\\": joint,\\n        }\\n    return {\\n        \\"status\\": \\"PASS\\", \\"source\\": \\"Verified CSV training target_statistics, checked against label_vector observation masks\\",\\n        \\"study_count\\": len(train_rows), \\"labeled_study_count\\": len(masks), \\"targets\\": targets,\\n        \\"observed_label_cells\\": sum(stats[\\"observed\\"] for stats in targets.values()),\\n        \\"unknown_label_cells\\": sum(stats[\\"unknown\\"] for stats in targets.values()),\\n        \\"label_mask_policy\\": \\"Unknown and invalid cells remain unobserved; missing labels are never negative\\",\\n        \\"study_grouped_class_support\\": diagnostics,\\n        \\"patient_grouping\\": {\\"status\\": \\"NOT_ASSESSED\\", \\"reason\\": \\"No trustworthy dataset-wide study-to-patient mapping has been established; sampled PatientID metadata is insufficient\\"},\\n        \\"gold_resource\\": {\\"status\\": \\"NOT_ESTABLISHED\\", \\"reason\\": \\"Observed label counts do not establish an independent 58-study resource or its provenance\\"},\\n        \\"validation_design_status\\": \\"REVIEW_REQUIRED\\", \\"split_approved\\": False, \\"training_ready\\": False,\\n        \\"limitations\\": [\\n            \\"Sparse labels may prevent a reliable twelve-target validation estimate.\\",\\n            \\"Validation design must be reviewed before training.\\",\\n            \\"Necessary marginal positive/negative counts do not guarantee simultaneous class coverage across targets.\\",\\n            \\"A class-coverage witness does not establish balanced folds, patient independence or statistical reliability.\\",\\n            \\"No approved split, patient mapping, independent gold resource or performance estimate is created.\\",\\n        ],\\n    }\\n"}}')
_RSNA_PACKAGE = 'rsna_knee_phase_b_bundle'
for _module, _source in _RSNA_BUNDLE["sources"].items():
    _actual = hashlib.sha256(_source.encode("utf-8")).hexdigest()
    if _actual != _RSNA_BUNDLE["sha256"][_module + ".py"]:
        raise RuntimeError("Embedded source checksum mismatch: " + _module)
_RSNA_SOURCE_IDENTITY = {
    "kind": "embedded-notebook",
    "bundle_sha256": hashlib.sha256(json.dumps(_RSNA_BUNDLE, sort_keys=True, ensure_ascii=True, separators=(",", ":")).encode("utf-8")).hexdigest(),
    "source_sha256": dict(_RSNA_BUNDLE["sha256"]),
}

class _RsnaBundleLoader(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    package_name = _RSNA_PACKAGE

    def __init__(self, sources, fingerprint):
        # Bind this loader to its own bundle, not mutable notebook globals.
        self.sources = dict(sources)
        self.fingerprint = fingerprint

    def find_spec(self, fullname, path=None, target=None):
        if fullname == self.package_name:
            return importlib.util.spec_from_loader(fullname, self, is_package=True)
        prefix = self.package_name + "."
        if fullname.startswith(prefix):
            name = fullname[len(prefix):]
            if name in self.sources and name != "__init__":
                return importlib.util.spec_from_loader(fullname, self)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        name = "__init__" if module.__name__ == self.package_name else module.__name__.split(".")[-1]
        module.__file__ = "<embedded:" + name + ".py>"
        module.__rsna_bundle_sha256__ = self.fingerprint
        code = compile(self.sources[name], module.__file__, "exec")
        exec(code, module.__dict__)

# Preserve class identity when re-running an identical, verified bundle.
# Changed sources are reloaded and require the configuration cell to run again.
_rsna_fingerprint = _RSNA_SOURCE_IDENTITY["bundle_sha256"]
_rsna_cached = {
    name: module for name, module in sys.modules.items()
    if name == _RSNA_PACKAGE or name.startswith(_RSNA_PACKAGE + ".")
}
_rsna_same_bundle = (
    _RSNA_PACKAGE in _rsna_cached
    and all(getattr(module, "__rsna_bundle_sha256__", None) == _rsna_fingerprint
            for module in _rsna_cached.values())
)
if not _rsna_same_bundle:
    for _name in _rsna_cached:
        del sys.modules[_name]
    sys.meta_path[:] = [finder for finder in sys.meta_path if getattr(finder, "package_name", None) != _RSNA_PACKAGE]
    sys.meta_path.insert(0, _RsnaBundleLoader(_RSNA_BUNDLE["sources"], _rsna_fingerprint))
from rsna_knee_phase_b_bundle.phase_b_config import PhaseBConfig
from rsna_knee_phase_b_bundle.audit import run_preflight
from rsna_knee_phase_b_bundle.phase_b import run_phase_b, write_phase_b_report


In [ ]:
# Generated limits. Change project configuration and rebuild; no manual cells needed.
CONFIG = {'data_root': None,
 'input_base': '/kaggle/input',
 'inspect_csv': True,
 'max_csv_rows': 100000,
 'max_depth': 2,
 'max_dicom_file_bytes': 67108864,
 'max_dicom_seconds': 90,
 'max_directories': 128,
 'max_entries': 2048,
 'max_file_bytes': 33554432,
 'max_files_per_series': 8,
 'max_header_bytes': 2097152,
 'max_header_reads': 128,
 'max_inventory_depth': 6,
 'max_inventory_directories': 60000,
 'max_inventory_entries': 1200000,
 'max_inventory_seconds': 180,
 'max_pixel_decodes': 12,
 'max_pixel_elements': 4194304,
 'max_report_bytes': 1048576,
 'max_sample_file_entries': 4096,
 'max_sample_series': 32,
 'max_sample_total_entries': 32768,
 'output_dir': '/kaggle/working/rsna_audit'}
phase_b_config = PhaseBConfig.from_dict(CONFIG)


In [ ]:
if 'phase_b_config' not in globals() or not isinstance(phase_b_config, PhaseBConfig):
    raise RuntimeError('Missing or stale PhaseBConfig: use Run All to recreate the validated configuration.')
csv_preflight = run_preflight(phase_b_config.preflight_config(), source_identity=_RSNA_SOURCE_IDENTITY)
print(json.dumps({'csv_preflight_status': csv_preflight['status'], 'csv_contracts_status': csv_preflight['csv_contracts']['status']}))


In [ ]:
if 'csv_preflight' not in globals() or not isinstance(phase_b_config, PhaseBConfig):
    raise RuntimeError('Run the configuration and CSV preflight first, or use Run All.')
phase_b_report = run_phase_b(phase_b_config, source_identity=_RSNA_SOURCE_IDENTITY, preflight=csv_preflight)
print(json.dumps(phase_b_report, sort_keys=True, indent=2, allow_nan=False))


In [ ]:
if 'phase_b_report' not in globals() or not isinstance(phase_b_config, PhaseBConfig):
    raise RuntimeError('Run the audit first, or use Run All.')
if phase_b_report.get('source_identity') != _RSNA_SOURCE_IDENTITY or phase_b_report.get('configuration') != phase_b_config.to_dict():
    raise RuntimeError('The Phase B report is stale: use Run All before writing evidence.')
report_path = write_phase_b_report(phase_b_report, phase_b_config)
print(str(report_path))
